# Logistic Regression - Script Ordered Storyboard

This notebook mirrors the `script.md` narration for Chapter 1 (The Sigmoid).

Each scene exports either a PNG or GIF into `renders/` so shots can be sequenced directly in voice-over order.

- We intentionally skip the truck/car recap visuals.
- The required students `(3h study, 2h exam, passed)` and `(2h study, 3h exam, failed)` are included.
- Parallel lines at `ST - EL = 1, 2, 3` (and negatives) tie to the narrated `60%`, `80%`, `100%` pass proportions.


In [2]:
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D
from PIL import Image, ImageDraw

np.random.seed(7)

OUTPUT_DIR = Path("renders")
OUTPUT_DIR.mkdir(exist_ok=True)

# Canonical figure size and DPI for every render (matches sigmoid 3D 61–64).
EXPORT_FIGSIZE = (15.0, 9.5)
EXPORT_DPI = 200
EXPORT_ADJ = dict(left=0.04, right=0.96, bottom=0.06, top=0.96)
EXPORT_DUO_ADJ = dict(left=0.07, right=0.98, top=0.96, bottom=0.12)

# Typography / padding (single source of truth for exports); all scaled +25%.
FONT_SIZE = 11 * 1.25
AXIS_LABEL_SIZE = 12 * 1.25
LEGEND_SIZE = 10 * 1.25
TITLE_SIZE = 12 * 1.25
NOTE_SIZE = 10 * 1.25
ANNOT_SIZE = 9 * 1.25
SAVE_PAD_INCHES = 0.12

# Exponential scenes share the export canvas; margins tuned per plot (avoids bbox crop shift).
EXP_FIGSIZE = EXPORT_FIGSIZE
EXP_SUBPLOT_ADJ = dict(left=0.09, right=0.91, bottom=0.09, top=0.91)


def finalize_exponential_figure(fig):
    fig.subplots_adjust(**EXP_SUBPLOT_ADJ)


plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelpad": 8.0 * 1.25,
        "axes.titlepad": 10.0 * 1.25,
        "legend.fontsize": LEGEND_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.96,
        "legend.borderaxespad": 0.55,
        "legend.labelspacing": 0.35,
        "legend.handlelength": 1.35,
        "legend.handletextpad": 0.65,
        "savefig.pad_inches": SAVE_PAD_INCHES,
    }
)


PASS_COLOR = "#2ca02c"
FAIL_COLOR = "#d62728"
NEUTRAL_COLOR = "#4f4f4f"

CHECK_ICON_PATH = OUTPUT_DIR / "check.png"
CROSS_ICON_PATH = OUTPUT_DIR / "cross.png"


def _ensure_outcome_icons(size=120, line_width=15):
    _sc = size / 96.0
    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line(
        [(int(round(20 * _sc)), int(round(55 * _sc))), (int(round(42 * _sc)), int(round(76 * _sc))), (int(round(78 * _sc)), int(round(24 * _sc)))],
        fill=(44, 160, 44, 255),
        width=line_width,
    )
    img.save(CHECK_ICON_PATH)

    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.line([(int(round(24 * _sc)), int(round(24 * _sc))), (int(round(72 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    draw.line([(int(round(72 * _sc)), int(round(24 * _sc))), (int(round(24 * _sc)), int(round(72 * _sc)))], fill=(214, 39, 40, 255), width=line_width)
    img.save(CROSS_ICON_PATH)


_ensure_outcome_icons()
CHECK_ICON = np.asarray(Image.open(CHECK_ICON_PATH).convert("RGBA"))
CROSS_ICON = np.asarray(Image.open(CROSS_ICON_PATH).convert("RGBA"))


def add_outcome_icon(ax, x, y_value, passed, zoom=0.2, alpha=1.0, rotation_deg=0):
    image_array = CHECK_ICON if passed else CROSS_ICON
    if rotation_deg:
        # Match 3D icons: left–right flip then 180° rotation (odd half-turns of 180°)
        arr = np.asarray(image_array, dtype=np.uint8)
        if int(round(float(rotation_deg) / 180.0)) % 2:
            arr = np.flip(arr, axis=1)
            arr = np.rot90(arr, k=2, axes=(0, 1))
        image_array = arr
    icon = OffsetImage(image_array, zoom=zoom)
    icon.set_alpha(alpha)
    ab = AnnotationBbox(icon, (x, y_value), frameon=False)
    ax.add_artist(ab)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def fig_to_image(fig, dpi=None, tight_layout=False):
    if dpi is None:
        dpi = EXPORT_DPI
    buf = io.BytesIO()
    kw = {"format": "png", "dpi": dpi, "pad_inches": SAVE_PAD_INCHES}
    if tight_layout:
        kw["bbox_inches"] = "tight"
    else:
        kw["bbox_inches"] = None
    fig.savefig(buf, **kw)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def save_gif(images, filename, duration=40):
    if not images:
        raise ValueError("images list is empty")
    images[0].save(
        OUTPUT_DIR / filename,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
    )


In [3]:
# Two-stage dataset design:
# 1) Before "that's not realistic": linearly separable points.
# 2) After that line: add symmetric noisy students to break separability.

separable_points = [
    # Fail class (left of boundary, z=study-exam < 0)
    (2, 3, 0),  # required fail example
    (4, 5, 0), (5, 6, 0),
    (1, 3, 0), (2, 4, 0), (4, 6, 0),
    (1, 4, 0), (3, 6, 0), (1, 6, 0),


    # Pass class (right of boundary, z > 0)
    (3, 2, 1),  # required pass example
    (5, 4, 1), (6, 5, 1),                    # z=+1 -> 3 pass
    (4, 2, 1), (6, 4, 1), (3, 1, 1),  # z=+2 -> 4 pass
    (4, 1, 1), (5, 2, 1), (6, 3, 1),  # z=+3 -> 4 pass
    (6, 2, 1),  # z=+4 (Scene 6b anchor)
    (6, 1, 1),
]

# Added noisy students (symmetric around z=0, on parallel lines)
# to break strict linear separability after "but that's not realistic".
noisy_symmetric_points = [
    # symmetric pairs on +/-1
    (2, 1, 0),  # z=+1 but fails
    (1, 2, 1),  # z=-1 but passes
    (3, 4, 1),  # z=-2 but fails
    (4, 3, 0),  # z=+2 but passes

    # symmetric pair on +/-2
    (3, 5, 1),  # z=+2 but passes
    (5, 3, 0),  # z=-2 but fails
]

realistic_points = separable_points + noisy_symmetric_points


def unpack_points(point_list):
    arr = np.array(point_list, dtype=float)
    s = arr[:, 0]
    e = arr[:, 1]
    labels = arr[:, 2].astype(int)
    diff = s - e
    return s, e, labels, diff


study_sep, exam_sep, y_sep, z_sep = unpack_points(separable_points)
study_real, exam_real, y_real, z_real = unpack_points(realistic_points)

xlim = (0, 7)
ylim = (0, 7)

midpoint_shift = (z_sep[y_sep == 0].max() + z_sep[y_sep == 1].min()) / 2.0

print(f"Separable students: {len(separable_points)}")
print(f"Realistic students: {len(realistic_points)}")
print("Required pass example present:", np.any((study_sep == 3) & (exam_sep == 2) & (y_sep == 1)))
print("Required fail example present:", np.any((study_sep == 2) & (exam_sep == 3) & (y_sep == 0)))
print("Threshold midpoint between classes (separable stage):", midpoint_shift)


Separable students: 20
Realistic students: 26
Required pass example present: True
Required fail example present: True
Threshold midpoint between classes (separable stage): 0.0


## Scene 1 - Introduce the dataset

Script alignment: survey students, collect study time, exam length, and pass/fail outcome.

The **next code cell** builds the **`00_student_survey_table.gif`** survey: header row **Student / Exam length / Study time / Pass / fail**, then the **full realistic** roster filled **one cell at a time** (✓ / ✗ outcomes). After the **helper** cell (`draw_dataset`, etc.), the **intro montage** cell exports **`01`–`17`** (axis-label emphasis, plain axes, staged collection order, boundary crossing, then threshold visuals). The **cells after that** export the **full** realistic dataset with **ST - EL = 0** and the marker view (`18`–`19`), then separable overviews (`20`–`21`).


In [87]:
# --- 00: survey table — header cells then each student's exam, study, and outcome one cell at a time (full realistic set).
TABLE_MS = 36
HEADER_CELL_HOLD = 2
DATA_CELL_HOLD = 1
TABLE_END_HOLD = 5
SURVEY_TABLE_FIGSIZE = (EXPORT_FIGSIZE[0] * 0.5, EXPORT_FIGSIZE[1])

_headers = ["Student", "Exam length (h)", "Study time (h)", "Pass / fail"]
_order = np.lexsort((exam_real, study_real))
_n = int(len(_order))

_full = [list(_headers)]
for _k, _idx in enumerate(_order, start=1):
    _s = float(study_real[_idx])
    _e = float(exam_real[_idx])
    _y = int(y_real[_idx])
    _full.append(
        [
            str(_k),
            f"{_e:g}",
            f"{_s:g}",
            "✓" if _y == 1 else "✗",
        ]
    )

_cells_order = []
for _c in range(4):
    _cells_order.append((0, _c))
for _r in range(1, _n + 1):
    for _c in range(4):
        _cells_order.append((_r, _c))


def _survey_table_frame(ax, n_visible_cells: int):
    ax.clear()
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    disp = [[""] * 4 for _ in range(_n + 1)]
    for _t in range(min(n_visible_cells, len(_cells_order))):
        rr, cc = _cells_order[_t]
        disp[rr][cc] = _full[rr][cc]
    tbl = ax.table(
        cellText=disp,
        loc="center",
        cellLoc="center",
        edges="closed",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(FONT_SIZE)
    tbl.scale(1.15, 2.05)
    for (_r, _c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#2a2a2a")
        cell.set_linewidth(0.85)
        if _r == 0:
            cell.set_facecolor("#e8e8e8")
            cell.get_text().set_fontweight("bold")
        else:
            cell.set_facecolor("#fafafa")
        txt = disp[_r][_c]
        if txt in ("✓",):
            cell.get_text().set_color(PASS_COLOR)
            cell.get_text().set_fontweight("bold")
        elif txt in ("✗",):
            cell.get_text().set_color(FAIL_COLOR)
            cell.get_text().set_fontweight("bold")
    fig = ax.figure
    fig.subplots_adjust(left=0.04, right=0.96, top=0.96, bottom=0.04)


def _survey_table_fig_to_pil(fig):
    """Tight bbox so the table is never clipped; caller closes fig."""
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=max(float(SAVE_PAD_INCHES), 0.28),
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    buf.seek(0)
    return Image.open(buf).convert("RGB")


_raw_survey = []
for _nv in range(len(_cells_order) + 1):
    _hold = HEADER_CELL_HOLD if _nv <= 4 else DATA_CELL_HOLD
    if _nv == len(_cells_order):
        _hold = TABLE_END_HOLD
    for _ in range(_hold):
        fig, ax = plt.subplots(figsize=SURVEY_TABLE_FIGSIZE)
        _survey_table_frame(ax, _nv)
        _raw_survey.append(_survey_table_fig_to_pil(fig))
        plt.close(fig)

_sw = max(im.size[0] for im in _raw_survey)
_sh = max(im.size[1] for im in _raw_survey)
frames_survey = []
for im in _raw_survey:
    canvas = Image.new("RGB", (_sw, _sh), (255, 255, 255))
    canvas.paste(im, ((_sw - im.width) // 2, (_sh - im.height) // 2))
    frames_survey.append(canvas)

save_gif(frames_survey, "00_student_survey_table.gif", duration=TABLE_MS)


In [5]:
def draw_dataset(ax, study_vals, exam_vals, labels, mask=None, alpha=0.95, title=None):
    if mask is None:
        mask = np.ones(len(study_vals), dtype=bool)

    for i in np.where(mask)[0]:
        add_outcome_icon(ax, study_vals[i], exam_vals[i], passed=bool(labels[i]), alpha=alpha)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def add_combined_legend(ax, loc="upper left"):
    line_handles, line_labels = ax.get_legend_handles_labels()
    merged_handles = []
    merged_labels = []
    seen = set()

    for h, lbl in zip(line_handles, line_labels):
        if lbl and lbl not in seen:
            merged_handles.append(h)
            merged_labels.append(lbl)
            seen.add(lbl)

    if merged_handles:
        ax.legend(
            handles=merged_handles,
            labels=merged_labels,
            loc=loc,
            prop={"size": LEGEND_SIZE},
        )


def add_threshold_line(ax, shift=0.0, label=None, style="--", color=NEUTRAL_COLOR, linewidth=1.0, x_range=None):
    x0, x1 = x_range if x_range is not None else xlim
    x = np.linspace(x0, x1, 200)
    y_line = x - shift
    ax.plot(x, y_line, style, color=color, linewidth=linewidth, label=label)



def draw_axes_only(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def draw_axes_only_study_bold(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(
        "Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10, fontweight="bold"
    )
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def draw_axes_only_exam_bold(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel(
        "Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10, fontweight="bold"
    )
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def shade_pass_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    top = np.minimum(y_line, yb)
    bot = np.full_like(xs, ya)
    ax.fill_between(xs, bot, top, where=(top > bot), alpha=alpha, color=PASS_COLOR, linewidth=0, zorder=0)


def shade_fail_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    bot2 = np.maximum(y_line, ya)
    ax.fill_between(xs, bot2, yb, where=(yb > bot2), alpha=alpha, color=FAIL_COLOR, linewidth=0, zorder=0)


def draw_z_axis_panel(ax, z_value, label):
    ax.axhline(0, color="black", linewidth=1)
    ax.scatter([z_value], [0], color="#ff7f0e", s=150, zorder=5)
    ax.text(z_value + 0.05, 0.08, label, fontsize=NOTE_SIZE)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.tick_params(axis="x", which="major", labelsize=FONT_SIZE)
    ax.grid(alpha=0.15)


def save_fig(fig, filename):
    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=EXPORT_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


In [38]:
# Intro montage (exports 01–17): label beats, axes-only, staged build, crossing story, threshold pedagogy.
rng_build = np.random.default_rng(7)
n_sep = len(study_sep)

AXIS_LABEL_HOLD = 22

# ---- 00: axes only — Study time label bold (no data) ----
frames_axis_labels = []
for _ in range(AXIS_LABEL_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_axes_only_study_bold(ax)
    frames_axis_labels.append(fig_to_image(fig))
save_gif(frames_axis_labels, "01_axes_study_time_bold.gif", duration=35)

# ---- 01: axes only — Exam length label bold (no data) ----
frames_axis_labels = []
for _ in range(AXIS_LABEL_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_axes_only_exam_bold(ax)
    frames_axis_labels.append(fig_to_image(fig))
save_gif(frames_axis_labels, "02_axes_exam_length_bold.gif", duration=35)

# ---- 02: axes only (no data), both labels regular weight ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_axes_only(ax)
save_fig(fig, "03_axes_only.png")

# ---- 03: dataset build — (3,2,1) then (2,3,0) then random order ----
idx_pass_req = int(np.where((study_sep == 3) & (exam_sep == 2) & (y_sep == 1))[0][0])
idx_fail_req = int(np.where((study_sep == 2) & (exam_sep == 3) & (y_sep == 0))[0][0])
rest = [i for i in range(n_sep) if i not in (idx_pass_req, idx_fail_req)]
rng_build.shuffle(rest)
reveal_order = [idx_pass_req, idx_fail_req] + rest

frames = []
# Hold empty axes, then one new point per step; first two points linger much longer.
DATASET_EMPTY_HOLD = 24
DATASET_FIRST_TWO_HOLD = 48
DATASET_REST_HOLD = 2


def _append_dataset_build_frames(mask, n_repeat):
    for _ in range(n_repeat):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=mask)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))


mask = np.zeros(n_sep, dtype=bool)
_append_dataset_build_frames(mask, DATASET_EMPTY_HOLD)

for k in range(1, n_sep + 1):
    mask = np.zeros(n_sep, dtype=bool)
    mask[reveal_order[:k]] = True
    hold = DATASET_FIRST_TWO_HOLD if k <= 2 else DATASET_REST_HOLD
    _append_dataset_build_frames(mask, hold)

save_gif(frames, "04_dataset_build.gif", duration=35)

# ---- 02: failed point pushed right (more study); threshold/boundary not drawn ----
EXAM_ANIM = 3.0
ST_START, ST_END = 2.0, 3.25
N_ANIM = 55
PUSH_HOLD = 5
st_values = np.concatenate([np.linspace(ST_START, ST_END, N_ANIM), np.full(PUSH_HOLD, ST_END)])
static_mask = np.ones(n_sep, dtype=bool)
static_mask[idx_fail_req] = False
frames = []
for st in st_values:
    passed = (st - EXAM_ANIM) > 1e-6
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
    add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.3, alpha=1.0)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "05_fail_point_crosses_threshold.gif", duration=42)

# ---- 05: threshold, unlabeled ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "06_threshold_unlabeled.png")

# ---- 06: unlabeled threshold + pass half green ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "07_threshold_green_right.png")

# ---- 07: unlabeled threshold + fail half red ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "08_threshold_red_left.png")

# ---- 08: threshold drifts slightly (still unlabeled) ----
drift = np.linspace(-0.99, 0.99, 100)
frames = []
for sh in drift:
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift + sh, label=None, color="black", linewidth=1)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "09_threshold_micro_drift.gif", duration=40)

# ---- 09: add (3,3) pass; threshold shifted so classes still separated ----
SHIFT_FOR_33 = -0.28
st_33 = np.append(study_sep, 3.0)
ex_33 = np.append(exam_sep, 3.0)
y_33 = np.append(y_sep, 1)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, st_33, ex_33, y_33)
add_threshold_line(ax, shift=midpoint_shift + SHIFT_FOR_33, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "10_dataset_extra_pass_shifted_threshold.png")

# ---- 10: (3,3) pass + (4,3) fail, no threshold ----
st_3343 = np.append(st_33, 4.0)
ex_3343 = np.append(ex_33, 3.0)
y_3343 = np.append(y_33, 0)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, st_3343, ex_3343, y_3343)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "11_dataset_two_extra_no_threshold.png")

# ---- 11: threshold labeled ST = EL ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "12_threshold_labeled_st_eq_el.png")

# ---- 12: ST = EL + black circle at (3,3) ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.plot([3], [3], marker="o", markersize=14, markerfacecolor="black", markeredgecolor="black", linestyle="none", zorder=6)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "13_threshold_st_eq_el_dot_33.png")

# ---- 13: labeled threshold + green pass half + ST > EL ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(4.5, 3.0, "ST > EL", fontsize=AXIS_LABEL_SIZE, color="#145214", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "14_threshold_green_labeled_st_gt_el.png")

# ---- 14: labeled threshold + red fail half + ST < EL ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(2.0, 5.0, "ST < EL", fontsize=AXIS_LABEL_SIZE, color="#7a1212", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "15_threshold_red_labeled_st_lt_el.png")

# ---- 15: line labeled ST - EL = EL - EL ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = EL - EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "16_threshold_label_st_minus_el_eq_el_minus_el.png")

# ---- 16: line labeled ST - EL = 0 ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "17_threshold_label_st_minus_el_eq_0.png")


In [39]:
# Full realistic dataset (all examples): ST - EL = 0, then same with boundary marker at (3, 3).
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_real, exam_real, y_real)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "18_dataset_full_realistic_st_el_0.png")

fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_real, exam_real, y_real)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
ax.plot(
    [3],
    [3],
    marker="o",
    markersize=14,
    markerfacecolor="black",
    markeredgecolor="black",
    linestyle="none",
    zorder=6,
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "19_dataset_full_realistic_st_el_0_marker_33.png")


In [40]:
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "20_dataset_overview.png")


In [41]:
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"ST - EL = {midpoint_shift:.1f}", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "21_dataset_overview_labeled.png")


## Scene 4 - Parallel lines (separate additive plots)

Script alignment: **`23`** animates markers on the **ST - EL = 0** diagonal from **(1,1)** through **(6,6)**; **`26`** highlights **(2,1)** with the same black marker style, then bold ticks **2** (study time) and **1** (exam length); then each line for fixed `ST - EL` appears one-by-one in additive order (**0, 1, 2, 3, 4**; no **-1** line). **`32_fail_point_crosses_threshold_st_el_0.gif`** replays the intro fail-point push (**`04_dataset_build.gif`**) with **ST - EL = 0** drawn.


In [70]:
# ---- 19: GIF — dataset + ST-EL=0 line; black markers appear on diagonal (1,1)…(6,6) one at a time ----
DIAG_PT_HOLD = 4
DIAG_BASE_HOLD = 3
DIAG_MS = 40
diag_pts = [(float(i), float(i)) for i in range(1, 7)]

frames19 = []

for _ in range(DIAG_BASE_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    add_combined_legend(ax, loc="upper left")
    frames19.append(fig_to_image(fig))

for k in range(1, len(diag_pts) + 1):
    for _ in range(DIAG_PT_HOLD):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        draw_dataset(ax, study_sep, exam_sep, y_sep)
        add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
        for tx, ty in diag_pts[:k]:
            ax.plot(
                [tx],
                [ty],
                marker="o",
                markersize=14,
                markerfacecolor="black",
                markeredgecolor="black",
                linestyle="none",
                zorder=6,
            )
        add_combined_legend(ax, loc="upper left")
        frames19.append(fig_to_image(fig))

save_gif(frames19, "22_dataset_points_on_threshold.gif", duration=DIAG_MS)

# ---- 20–21: same half-plane emphasis as 13/14, labels ST - EL > 0 / ST - EL < 0 ----
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
ax.text(
    4.5,
    3.0,
    "ST - EL > 0",
    fontsize=AXIS_LABEL_SIZE,
    color="#145214",
    fontweight="bold",
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "23_threshold_green_st_minus_el_gt_0.png")

fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
ax.text(
    2.0,
    5.0,
    "ST - EL < 0",
    fontsize=AXIS_LABEL_SIZE,
    color="#7a1212",
    fontweight="bold",
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "24_threshold_red_st_minus_el_lt_0.png")

# ---- 22: GIF — separable dataset + threshold; black marker at (2,1); bold ST tick 2 then EL tick 1 ----
HIGHLIGHT_MS = 40
HIGHLIGHT_HOLD = 5


def _bold_tick_at_value(ax, axis, value):
    labels = ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()
    for lbl in labels:
        t = lbl.get_text().strip()
        try:
            if abs(float(t) - float(value)) < 1e-9:
                lbl.set_fontweight("bold")
        except ValueError:
            continue


def _frame_dataset_highlight(ax, *, show_marker, bold_st2, bold_el1):
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    if show_marker:
        ax.plot(
            [2.0],
            [1.0],
            marker="o",
            markersize=14,
            markerfacecolor="black",
            markeredgecolor="black",
            linestyle="none",
            zorder=6,
        )
    add_combined_legend(ax, loc="upper left")
    if bold_st2:
        _bold_tick_at_value(ax, "x", 2)
    if bold_el1:
        _bold_tick_at_value(ax, "y", 1)


frames_highlight = []
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    _frame_dataset_highlight(ax, show_marker=False, bold_st2=False, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=False, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=True, bold_el1=False)
    frames_highlight.append(fig_to_image(fig))
for _ in range(HIGHLIGHT_HOLD):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    _frame_dataset_highlight(ax, show_marker=True, bold_st2=True, bold_el1=True)
    frames_highlight.append(fig_to_image(fig))

save_gif(frames_highlight, "25_dataset_st2_el1_marker_ticks.gif", duration=HIGHLIGHT_MS)

line_specs = [
    (0, "black", "ST - EL = 0"),
    (1, "#1f77b4", "ST - EL = 1"),
    (-1, "#17becf", "ST - EL = -1"),
    (2, "#2ca02c", "ST - EL = 2"),
    (3, "#bcbd22", "ST - EL = 3"),
]

PARALLEL_HOLD = 4

frames = []
for step in range(1, len(line_specs) + 1):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep)

    for shift, color, label in line_specs[:step]:
        add_threshold_line(ax, shift=shift, label=label, color=color, linewidth=1)

    add_combined_legend(ax, loc="upper left")
    file_name = f"{25 + step:02d}_parallel_lines_step_{step:02d}.png"
    save_fig(fig, file_name)

    frame = Image.open(OUTPUT_DIR / file_name).convert("RGB")
    for _ in range(PARALLEL_HOLD):
        frames.append(frame)

save_gif(frames, "31_parallel_lines_additive.gif", duration=40)

# ---- 25: like `05_fail_point_crosses_threshold.gif` but draw ST - EL = 0 (same motion at EL=3) ----
frames = []
for st in st_values:
    passed = (st - EXAM_ANIM) > 1e-6
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.3, alpha=1.0)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "32_fail_point_crosses_threshold_st_el_0.gif", duration=42)


## Scene 5 - "But that's not realistic" (add symmetric noisy students)

Script alignment:
- realistic students break perfect separability while keeping the same midpoint threshold
- `33_not_realistic_transition.gif`: **separable** dataset + threshold first; only the **new** (noisy) students appear one at a time — first **(4,3)** then **(3,4)**, then the rest (axis ticks/labels + legend like other dataset plots)
- `34_sigmoid_colormap_topdown_reveal.gif`: top-down **σ(0.5·ST − 0.5·EL)** colormap (same style as the final static top-down PNG): smooth reveal of **ST − EL > 0**, then **ST − EL < 0**, with threshold and outcome icons
- `35_parallel_lines_additive_realistic.gif`: same additive cadence as **`31_parallel_lines_additive.gif`**, on the **full realistic** dataset; only **`ST - EL = 0`** is legend-labeled; colored guides **`ST - EL = 1, -1, 2, 3`** in that order (no `z=4` line)
- `36_proportions_60_80_100.gif`: **full** realistic dataset + **black** threshold throughout; colored lines **`ST - EL = 1, -1, 2, 3`** (same order) and pass-rate text animate **on top** (same counting beats for **+1** as before; same axes/legend as other dataset plots)
- narrated proportions for `ST - EL = 1, 2, 3` remain `60%`, `80%`, `100%`


In [43]:
# Scene 5 — "not realistic": separable dataset + threshold first; noisy points one-by-one: (4,3) then (3,4) then lexsorted rest; full axis labels/ticks + legend.
SC5_MS = 40
HOLD_BASELINE = 18
HOLD_PER_POINT = 2
n_r = len(study_real)
n_sep = len(study_sep)
new_idx = np.arange(n_sep, n_r)
priority_pairs = ((4.0, 3.0), (3.0, 4.0))
priority = []
for s0, e0 in priority_pairs:
    hit = np.where((study_real == s0) & (exam_real == e0))[0]
    if hit.size:
        priority.append(int(hit[0]))
rest = [int(i) for i in new_idx if int(i) not in priority]
rest_arr = np.array(rest, dtype=int)
if rest_arr.size:
    rest_sorted = rest_arr[np.lexsort((exam_real[rest_arr], study_real[rest_arr]))].tolist()
else:
    rest_sorted = []
reveal_new = np.array(priority + rest_sorted, dtype=int)

frames = []
for _ in range(HOLD_BASELINE):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    mask_sep = np.zeros(n_r, dtype=bool)
    mask_sep[:n_sep] = True
    draw_dataset(ax, study_real, exam_real, y_real, mask=mask_sep, alpha=0.95)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))

for k in range(1, len(reveal_new) + 1):
    mask = np.zeros(n_r, dtype=bool)
    mask[:n_sep] = True
    mask[reveal_new[:k]] = True
    for _ in range(HOLD_PER_POINT):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.grid(alpha=0.12)
        add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
        draw_dataset(ax, study_real, exam_real, y_real, mask=mask, alpha=0.95)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))

save_gif(frames, "33_not_realistic_transition.gif", duration=SC5_MS)

# ---- 27: top-down colormap σ(0.5·ST − 0.5·EL) — smooth reveal ST−EL > 0, then ST−EL < 0 (same layout as final top-down PNG) ----
import matplotlib as mpl

TOPDOWN_REVEAL_MS = 42
GRID_TD = 220
st_td = np.linspace(xlim[0], xlim[1], GRID_TD)
el_td = np.linspace(ylim[0], ylim[1], GRID_TD)
ST_td, EL_td = np.meshgrid(st_td, el_td)
DIFF_td = ST_td - EL_td
Z_td = sigmoid(0.5 * ST_td - 0.5 * EL_td)
cvals_td = [0, 0.5, 1]
colors_td = ["red", "white", "green"]
norm_td = plt.Normalize(min(cvals_td), max(cvals_td))
tuples_td = list(zip(map(norm_td, cvals_td), colors_td))
CMAP_td = mpl.colors.LinearSegmentedColormap.from_list("", tuples_td, 100)
dmax_td = float(np.max(DIFF_td))
dmin_td = float(np.min(DIFF_td))
eps_td = 0.05 * max(dmax_td, abs(dmin_td))


def _ease_smooth(t):
    t = np.clip(t, 0.0, 1.0)
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _fig_topdown_masked(zm):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    ax.contourf(
        ST_td,
        EL_td,
        zm,
        levels=np.linspace(0.0, 1.0, 45),
        cmap=CMAP_td,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.32,
    )
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    for s, e, lbl in zip(study_real, exam_real, y_real):
        add_outcome_icon(ax, float(s), float(e), passed=bool(lbl), zoom=0.2, alpha=0.95)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.12)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")
    return fig_to_image(fig)


frames_td = []
N_PASS = 52
N_FAIL = 52
HOLD_PASS_FULL = 10
HOLD_ALL_FULL = 14

for j in range(N_PASS):
    u = eps_td + (dmax_td - eps_td) * _ease_smooth(j / max(N_PASS - 1, 1))
    zm = np.where((DIFF_td > 0) & (DIFF_td <= u), Z_td, np.nan)
    frames_td.append(_fig_topdown_masked(zm))

zm_pass_done = np.where(DIFF_td > 0, Z_td, np.nan)
for _ in range(HOLD_PASS_FULL):
    frames_td.append(_fig_topdown_masked(zm_pass_done))

for j in range(N_FAIL):
    lo = -eps_td + (dmin_td + eps_td) * _ease_smooth(j / max(N_FAIL - 1, 1))
    zm = np.where(
        DIFF_td > 0,
        Z_td,
        np.where((DIFF_td >= lo) & (DIFF_td < 0), Z_td, np.nan),
    )
    frames_td.append(_fig_topdown_masked(zm))

zm_all = Z_td
for _ in range(HOLD_ALL_FULL):
    frames_td.append(_fig_topdown_masked(zm_all))

save_gif(frames_td, "34_sigmoid_colormap_topdown_reveal.gif", duration=TOPDOWN_REVEAL_MS)

# ---- 28: like `31_parallel_lines_additive.gif` but full **realistic** dataset; only z=0 labeled; guides z ∈ {1,-1,2,3} (no z=4) ----
line_specs_realistic = [
    (0, "black", "ST - EL = 0"),
    (1, "#1f77b4", None),
    (2, "#17becf", None),
    (3, "#2ca02c", None),
]
PARALLEL_REAL_MS = 40
PARALLEL_REAL_HOLD = 4
frames_pr = []
for step in range(1, len(line_specs_realistic) + 1):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    draw_dataset(ax, study_real, exam_real, y_real, mask=np.ones(n_r, dtype=bool), alpha=0.95)
    for shift, color, lbl in line_specs_realistic[:step]:
        add_threshold_line(ax, shift=shift, label=lbl, color=color, linewidth=1)
    add_combined_legend(ax, loc="upper left")
    im_pr = fig_to_image(fig)
    for _ in range(PARALLEL_REAL_HOLD):
        frames_pr.append(im_pr.copy())

save_gif(frames_pr, "35_parallel_lines_additive_realistic.gif", duration=PARALLEL_REAL_MS)

# --- Line-level proportions (realistic): lines accumulate **1 → -1 → 2 → 3**; +1 uses grow/shrink count. ---
targets = [1, 2, 3]
line_colors = ["#1f77b4", "#17becf", "#2ca02c"]
prop = []
counts = []
for value in targets:
    m = z_real == value
    prop.append(float(y_real[m].mean()) if m.any() else 0.0)
    counts.append(int(m.sum()))

frames = []


def _prop_axes_backdrop(ax):
    """Full realistic plane: limits, black threshold, all students, axis labels/ticks + legend."""
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    draw_dataset(
        ax, study_real, exam_real, y_real, mask=np.ones(n_r, dtype=bool), alpha=0.95
    )
    add_combined_legend(ax, loc="upper left")


def _prob_xy_for_line(shift, x_hint=5.35):
    y_line = x_hint - shift
    return x_hint + 0.12, y_line + 0.28


def _emit_hold(img, n):
    for _ in range(n):
        frames.append(img.copy())


lines_pairs = []

for _ in range(10):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    _prop_axes_backdrop(ax)
    frames.append(fig_to_image(fig))

for ti, (value, lcol) in enumerate(zip(targets, line_colors)):
    lines_pairs.append((value, lcol))

    for _ in range(5):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        _prop_axes_backdrop(ax)
        for sh, col in lines_pairs:
            add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
        frames.append(fig_to_image(fig))

    if value == 1:
        idx1 = np.where(z_real == 1)[0]
        idx1 = idx1[np.argsort(study_real[idx1], kind="mergesort")]
        n1 = len(idx1)
        for j in range(0, n1 + 1):
            grow_states = (True, False) if j > 0 else (True,)
            for grow in grow_states:
                fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
                _prop_axes_backdrop(ax)
                for sh, col in lines_pairs:
                    add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
                for k, ii in enumerate(idx1):
                    if k >= j:
                        break
                    last = k == j - 1
                    zm = 0.325 if last and grow else 0.2
                    add_outcome_icon(ax, study_real[ii], exam_real[ii], passed=bool(y_real[ii]), zoom=zm, alpha=0.95)
                if j > 0:
                    pos = int(y_real[idx1[:j]].sum())
                    tx, ty = _prob_xy_for_line(1)
                    ax.text(tx, ty, f"{pos}/{j}", fontsize=NOTE_SIZE, color="black")
                im = fig_to_image(fig)
                frames.append(im)
                _emit_hold(im, 1 if grow else 0)
        p1, c1 = prop[ti], counts[ti]
        pos1 = int(round(p1 * c1))
        for _ in range(10):
            fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
            _prop_axes_backdrop(ax)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            tx, ty = _prob_xy_for_line(1)
            ax.text(tx, ty, f"{pos1}/{c1} = {100 * p1:.0f}%", fontsize=NOTE_SIZE, color="black")
            frames.append(fig_to_image(fig))
    else:
        idxv = np.where(z_real == value)[0]
        idxv = idxv[np.argsort(study_real[idxv], kind="mergesort")]
        for _ in range(8):
            fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
            _prop_axes_backdrop(ax)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            pv, cv = prop[ti], counts[ti]
            posv = int(round(pv * cv))
            tx, ty = _prob_xy_for_line(value)
            ax.text(tx, ty, f"{posv}/{cv} = {100 * pv:.0f}%", fontsize=NOTE_SIZE, color="black")
            frames.append(fig_to_image(fig))

save_gif(frames, "36_proportions_60_80_100.gif", duration=SC5_MS)

print("ST-EL=+1 pass rate:", prop[0])
print("ST-EL=+2 pass rate:", prop[1])
print("ST-EL=+3 pass rate:", prop[2])


ST-EL=+1 pass rate: 0.6
ST-EL=+2 pass rate: 0.75
ST-EL=+3 pass rate: 1.0


## Scene 6 - Student between 60% and 80%

Script alignment: a student with `ST - EL` between **+1** and **+2** should map to a probability between **0.6** and **0.8**.

`37_between_60_and_80.png` is the **study vs exam** plane (axis labels + legend like other dataset views): **`ST - EL = 0`** labeled in black; **`ST - EL = 1` / `2`** use the same blue/cyan as **`36_proportions_60_80_100.gif`** (that GIF also adds **`ST - EL = -1`** in orange between **+1** and **+2**); **black marker** at **(3.5, 2)** (`z = 1.5`).

`38_dataset_plane_into_norm_left_slot.gif` (same figure size as all exports) animates the **study vs exam** plane with **ST − EL = 0** only (no +1/+2 guide lines, no highlight marker), from a full **`EXPORT_ADJ`** canvas into the exact **left** subplot rectangle of the Scene **6b** duo (`width_ratios` **1.12 : 1.0**, **`wspace=0.28`**, **`EXPORT_DUO_ADJ`**), then holds so it lines up with **`39_norm_positive_two_bars.png`**.


In [44]:
# Scene 6 — "between" student: ST–EL plane with z = +1 / +2 guides (same colors as proportions `29`) + black marker at (3.5, 2); axis labels + threshold legend.
z_anchor = np.array([1.0, 2.0, 3.0])
p_anchor = np.array([0.6, 0.8, 1.0])

query_z = 1.5
query_p = np.interp(query_z, z_anchor, p_anchor)

fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.grid(alpha=0.12)
add_threshold_line(ax, shift=0, label="ST - EL = 0", color="black", linewidth=1)
add_threshold_line(ax, shift=1, label=None, color="#1f77b4", linewidth=1)
add_threshold_line(ax, shift=2, label=None, color="#17becf", linewidth=1)
draw_dataset(ax, study_real, exam_real, y_real, alpha=0.32)
ax.plot(
    [3.5],
    [2.0],
    marker="o",
    markersize=14,
    markerfacecolor="black",
    markeredgecolor="black",
    linestyle="none",
    zorder=15,
)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "37_between_60_and_80.png")

# --- 30b: study–exam plane with ST−EL=0 only (no +1/+2 lines, no marker); morph into Scene 6b **left** panel slot ---
_SCENE30B_MS = 42
_SCENE30B_MORPH = 56
_SCENE30B_HOLD = 14


def _draw_scene30_between(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=0, label="ST - EL = 0", color="black", linewidth=1)
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.32)
    add_combined_legend(ax, loc="upper left")


def _bbox_axes_in_fig(fig, ax):
    fig.canvas.draw()
    p = ax.get_position()
    return (float(p.x0), float(p.y0), float(p.width), float(p.height))


def _bbox_full_export_adj():
    tmp_fig, tmp_ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    tmp_fig.subplots_adjust(**EXPORT_ADJ)
    bb = _bbox_axes_in_fig(tmp_fig, tmp_ax)
    plt.close(tmp_fig)
    return bb


def _bbox_norm_duo_left():
    tmp_fig, (axl, axr) = plt.subplots(
        1,
        2,
        figsize=EXPORT_FIGSIZE,
        gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
    )
    tmp_fig.subplots_adjust(**EXPORT_DUO_ADJ)
    bb = _bbox_axes_in_fig(tmp_fig, axl)
    plt.close(tmp_fig)
    return bb


def _smooth01(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


bb0 = _bbox_full_export_adj()
bb1 = _bbox_norm_duo_left()
frames_30b = []
for j in range(_SCENE30B_MORPH):
    u = _smooth01(j / (_SCENE30B_MORPH - 1)) if _SCENE30B_MORPH > 1 else 1.0
    l = bb0[0] + u * (bb1[0] - bb0[0])
    b = bb0[1] + u * (bb1[1] - bb0[1])
    w = bb0[2] + u * (bb1[2] - bb0[2])
    h = bb0[3] + u * (bb1[3] - bb0[3])
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("white")
    ax = fig.add_axes([l, b, w, h])
    _draw_scene30_between(ax)
    frames_30b.append(fig_to_image(fig, tight_layout=False))
for _ in range(_SCENE30B_HOLD):
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    fig.patch.set_facecolor("white")
    ax = fig.add_axes(list(bb1))
    _draw_scene30_between(ax)
    frames_30b.append(fig_to_image(fig, tight_layout=False))

save_gif(frames_30b, "38_dataset_plane_into_norm_left_slot.gif", duration=_SCENE30B_MS)

print(f"Interpolated probability for ST-EL={query_z}: {query_p:.2f}")


Interpolated probability for ST-EL=1.5: 0.70


## Scene 6b - Normalization vs order (bridge into $2^x$)

Script alignment (`script.md` ~138–185): **realistic** plane (left) with **ST − EL = 0** threshold; **light green / dark green** (positive $z$) and **light red** ($z<0$) segments are perpendicular distances to that line. **Right:** vertical **ST − EL** bars at **A** / **B** ($y\in[-2,7]$, integer ticks). **B** anchored at **(6,2)**; **A** at **$z=+1$** pass and **$z=-1$** fail, chosen with separable labels and **not** on another student’s perpendicular segment to the threshold. GIFs step through share formulas **$1/(1{+}4)\!\to\!1/5\!\to\!20\%$** (and the **$(-1,4)$** / **$|z|$** variants). **`43_norm_abs_bar_hint.gif`/`44_norm_exp_bar_hint.gif`** walk integer **$z\in[-3,3]$** on the bars (on-line **$z=0$** uses a **black circle** on the plane). **`44b_norm_all_st_el_then_exp_emphasis.gif`** uses the **full** $z\in[-3,3]$ selection: **signed** bar heights (hold), then flips each bar **in order** to **$e^{\mathrm{ST}-\mathrm{EL}}$**, then the **effort** label, then **emphasizes** each exp bar while a **single curved arrow** grows **once** from the **smallest-$e^z$** bar top to the **largest-$e^z$** bar top (fixed $y\in[-4,22]$); **bold** edges walk bars in that same **smallest-to-largest** order. Same **red–white–green** colormap as **sigmoid 3D** (Scene 8) before **`46_exponential_mapping.gif`**.

**`41b_dataset_plane_2h_difference_segment.gif`** draws a **2 h** horizontal segment at **exam = 5 h** between **(4, 5)** and **(6, 5)** with a callout. **`41c_dataset_plane_zero_study_exam_gap_contrast.gif`** contrasts two **0 h study** cases (**1 h** vs **4 h** exam): same vertical in study–exam space, then perpendiculars to **ST − EL = 0** showing the **4 h exam** case lies **farther** below the threshold. Bridge **`38_dataset_plane_into_norm_left_slot.gif`**: the study–exam plane (dataset + **ST − EL = 0** only, matching the duo left panel) smoothly shrinks and slides into the **left** subplot slot used by **`39_norm_positive_two_bars.png`** (duo `width_ratios` / `wspace` / `EXPORT_DUO_ADJ`). Exports **`39_norm_positive_two_bars.png`–`44b_norm_all_st_el_then_exp_emphasis.gif`** (see Scene 9 checklist).

In [78]:
# Scene 6b — normalization bridge (exports 31–36): greens/reds, formula steps, exp bar colormap
NORM_MS = 42
NORM_DUO_FIGSIZE = EXPORT_FIGSIZE
COL_POS_LO = "#b8e8b8"
COL_POS_HI = "#1a6b1a"
COL_NEG_LO = "#f5bcbc"
N_BAR_YLIM = (-2, 7)
N_BAR_YTICKS = np.arange(-2, 8)
N_BAR_XLIM = (-0.75, 1.75)
N_STACK_X = 0.5
N_STACK_WIDTH = 0.5 * (N_BAR_XLIM[1] - N_BAR_XLIM[0])
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import patheffects
from matplotlib.path import Path as MplPath
from matplotlib.patches import FancyArrowPatch

# Stacked-bar formula labels (32–34): large type, always above bars; white text stroked for contrast on translucent red.
NORM_STACK_LABEL_FS = NOTE_SIZE + 14
NORM_STACK_LABEL_Z = 30
NORM_STACK_WHITE_PE = [patheffects.withStroke(linewidth=3.125, foreground="black")]
NORM_WORK_EFFORT_GLOSS_Z = 22

# Match Scene 8 sigmoid 3D `CMAP`: red (low) -> white (mid) -> green (high)
_EXPC = [0, 0.5, 1.0]
_EXPK = ["red", "white", "green"]
_EXPN = plt.Normalize(min(_EXPC), max(_EXPC))
_EXPT = list(zip(map(_EXPN, _EXPC), _EXPK))
_EXP_SIG_CMAP = LinearSegmentedColormap.from_list("exp_sig_rwg", _EXPT, 100)


def _exp_bar_facecolors(z_vals):
    out = []
    for z in z_vals:
        # Match sigmoid 3D semantics: z=0 -> white; z<0 redder, z>0 greener.
        t = float(z) / 6.0 + 0.5
        out.append(_EXP_SIG_CMAP(np.clip(t, 0.0, 1.0)))
    return out


def _norm_frame_png(fig):
    # Full figure bbox (no tight crop) so every GIF frame has identical pixel size.
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=EXPORT_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    buf.seek(0)
    im = Image.open(buf).convert("RGB").copy()
    buf.close()
    return im


def _threshold_foot(s, e, shift):
    t = (s + e + shift) / 2.0
    return t, t - shift


def _pick_realistic(s_target, e_target):
    m = np.isclose(study_real, s_target) & np.isclose(exam_real, e_target)
    idx = int(np.flatnonzero(m)[0])
    return idx, float(study_real[idx]), float(exam_real[idx]), float(z_real[idx]), int(y_real[idx])


def _draw_plane_base(ax):
    # Clear each frame so repeated threshold draws do not stack (linewidth creep in GIFs).
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label="ST - EL = 0",
        color="black",
        linewidth=1,
        x_range=tuple(ax.get_xlim()),
    )
    add_combined_legend(ax, loc="upper left")


def _draw_student_distance(ax, s, e, y, color):
    fs, fe = _threshold_foot(s, e, midpoint_shift)
    ax.plot([s, fs], [e, fe], color=color, linewidth=2.8, solid_capstyle="round", zorder=4)
    add_outcome_icon(ax, s, e, passed=bool(y), alpha=0.98, zoom=0.2)


def _norm6b_on_perp_open(px, py, qx, qy, shift, eps=1e-5):
    """True if Q lies strictly between P and the foot of the perpendicular from P to ST-EL = shift."""
    P = np.array([px, py], dtype=float)
    Q = np.array([qx, qy], dtype=float)
    t = (px + py + shift) / 2.0
    F = np.array([t, t - shift], dtype=float)
    v = F - P
    nv = float(np.hypot(v[0], v[1]))
    if nv < eps:
        return False
    w = Q - P
    if abs(w[0] * v[1] - w[1] * v[0]) > eps * nv:
        return False
    tv = float(np.dot(v, v))
    if tv < eps * eps:
        return False
    tparam = float(np.dot(w, v) / tv)
    return eps < tparam < 1.0 - eps


def _draw_barhint_plane_marker(ax, s, e, y, z, color):
    fs, fe = _threshold_foot(s, e, midpoint_shift)
    ax.plot([s, fs], [e, fe], color=color, linewidth=2.8, solid_capstyle="round", zorder=4)
    if np.isclose(z, 0.0):
        ax.plot(
            [s],
            [e],
            linestyle="none",
            marker="o",
            markersize=14,
            markerfacecolor="black",
            markeredgecolor="black",
            zorder=8,
        )
    else:
        add_outcome_icon(ax, s, e, passed=bool(y), alpha=0.98, zoom=0.2)


def _norm6b_pick_index_for_z(zt, want_y, ref_s, ref_e):
    idxs = np.flatnonzero(np.isclose(z_real, zt, atol=1e-9) & (y_real == int(want_y)))
    if not idxs.size:
        idxs = np.flatnonzero(np.isclose(z_real, zt, atol=1e-9))
    sh = float(midpoint_shift)
    best_i = None
    best_key = None
    relax_i = None
    relax_key = None
    for i in idxs:
        s = float(study_real[i])
        ee = float(exam_real[i])
        ok = (not _norm6b_on_perp_open(ref_s, ref_e, s, ee, sh)) and (not _norm6b_on_perp_open(s, ee, ref_s, ref_e, sh))
        md = float(np.hypot(s - ref_s, ee - ref_e))
        key = (md, -s)
        if relax_i is None or key > relax_key:
            relax_key, relax_i = key, int(i)
        if ok and (best_i is None or key > best_key):
            best_key, best_i = key, int(i)
    return int(best_i if best_i is not None else relax_i)


def _draw_left_progress(ax, s_a, e_a, y_a, c_a, s_b, e_b, y_b, c_b, phase):
    # phase: 0 none, 1 A only, 2 A+B (B then A so A's perpendicular stays on top when B is on ST-EL=0).
    _draw_plane_base(ax)
    if phase >= 2:
        _draw_student_distance(ax, s_b, e_b, y_b, c_b)
    if phase == 1:
        _draw_student_distance(ax, s_a, e_a, y_a, c_a)
    elif phase >= 2:
        _draw_student_distance(ax, s_a, e_a, y_a, c_a)


def _style_bar_axis(ax, stacked=False, ylabel=None, z_ylim=None):
    ylab = ylabel if ylabel is not None else "ST - EL"
    if z_ylim is None:
        ax.set_ylim(*N_BAR_YLIM)
        ax.set_yticks(N_BAR_YTICKS)
    else:
        ax.set_ylim(z_ylim[0], z_ylim[1])
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=8, prune="lower"))
    ax.set_ylabel(ylab, fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_xlabel("")
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set_xlim(*N_BAR_XLIM)
    if stacked:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.tick_params(axis="x", bottom=False, labelbottom=False)
    else:
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["A", "B"])


def _norm_work_effort_gloss(ax_r, kind):
    """In-axes callouts for script.md work vs effort (not new axis titles)."""
    bbox_kw = dict(
        boxstyle="round,pad=0.24",
        facecolor="white",
        edgecolor="#bbbbbb",
        linewidth=0.65,
        alpha=0.95,
    )
    if kind == "work":
        body = r"$\mathrm{work}=\mathrm{ST}-\mathrm{EL}$"
        xy, va = (0.04, 0.97), "top"
    elif kind == "effort":
        body = r"$\mathrm{effort}\propto e^{\mathrm{ST}-\mathrm{EL}}$"
        xy, va = (0.04, 0.97), "top"
    elif kind == "effort_stack":
        body = r"$\mathrm{effort}$ (stacked)"
        xy, va = (0.04, 0.97), "top"
    elif kind == "effort_normalize":
        body = (
            r"$\mathrm{effort}\propto e^{\mathrm{ST}-\mathrm{EL}}$"
            + "\n"
            + r"$\mathrm{normalize}\Rightarrow \mathrm{P}$"
        )
        xy, va = (0.04, 0.97), "top"
    elif kind == "pf_lane":
        body = (
            r"$\mathrm{work}=\mathrm{ST}-\mathrm{EL}$"
            + "\n"
            + r"$\mathrm{effort}\propto e^{\mathrm{work}}$"
        )
        xy, va = (0.04, 0.14), "bottom"
    else:
        return
    ax_r.text(
        xy[0],
        xy[1],
        body,
        transform=ax_r.transAxes,
        ha="left",
        va=va,
        fontsize=NOTE_SIZE,
        linespacing=1.28,
        color="#171717",
        bbox=bbox_kw,
        zorder=NORM_WORK_EFFORT_GLOSS_Z,
    )


def _draw_twin_bars(
    ax,
    z_a,
    z_b,
    c_a,
    c_b,
    phase,
    alpha_a=1.0,
    ylabel=None,
    z_ylim=None,
    work_effort_gloss=False,
    inline_stel_z_digits=False,
    stel_digits_b_only=False,
    stel_digit_white=False,
    stel_omit_zero_b=False,
):
    # phase: 0 none, 1 A only, 2 A+B
    ax.clear()
    if phase >= 1:
        ax.bar([0], [z_a], width=0.48, color=[c_a], edgecolor="black", linewidth=0.9, alpha=alpha_a)
    if phase >= 2:
        ax.bar([1], [z_b], width=0.48, color=[c_b], edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax, stacked=False, ylabel=ylabel, z_ylim=z_ylim)
    if work_effort_gloss:
        _norm_work_effort_gloss(ax, "work")
    if inline_stel_z_digits:
        _za = float(z_a)
        _zb = float(z_b)
        _fs = NOTE_SIZE + 12
        _txc = "#ffffff" if stel_digit_white else "#141414"
        _tpe = NORM_STACK_WHITE_PE if stel_digit_white else None
        _kw = dict(ha="center", va="center", fontsize=_fs, color=_txc, zorder=NORM_STACK_LABEL_Z)
        if _tpe is not None:
            _kw["path_effects"] = _tpe
        if stel_digits_b_only:
            if phase >= 2:
                _zbi = int(round(_zb))
                if not (stel_omit_zero_b and _zbi == 0):
                    ax.text(1.0, 0.5 * _zb, str(_zbi), **_kw)
        else:
            if phase >= 1:
                ax.text(0.0, 0.5 * _za, str(int(round(_za))), **_kw)
            if phase >= 2:
                _zbi = int(round(_zb))
                if not (stel_omit_zero_b and _zbi == 0):
                    ax.text(1.0, 0.5 * _zb, str(_zbi), **_kw)


def _draw_pos_stack_labels(ax_r, stage):
    h1, h2 = 1.0, 4.0
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$1/(1+4)$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$4/(1+4)$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$\frac{1}{5}$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$\frac{4}{5}$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 3:
        ax_r.text(N_STACK_X, 0.5 * h1, "20%", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            "80%",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )


def _draw_neg_stack_labels(ax_r, stage):
    zn, zp = -1.0, 4.0
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    kw = dict(ha="center", va="center", fontsize=fs, fontweight="bold", color="#ffffff", alpha=1.0, path_effects=NORM_STACK_WHITE_PE, zorder=zz)
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * zn, r"$\frac{-1}{-1+4}$", **kw)
        ax_r.text(N_STACK_X, zn + 0.5 * zp, r"$\frac{4}{-1+4}$", **kw)
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * zn, r"$-\frac{1}{3}$", **kw)
        ax_r.text(N_STACK_X, zn + 0.5 * zp, r"$\frac{4}{3}$", **kw)


def _with_pos_stack(ax_r, stage):
    h1, h2 = 1.0, 4.0
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=COL_POS_LO, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=COL_POS_HI, edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax_r, stacked=True)
    if stage > 0:
        _draw_pos_stack_labels(ax_r, stage)


def _with_neg_stack(ax_r, stage):
    zn, zp = -1.0, 4.0
    ax_r.clear()
    # Draw green first, then translucent red on top for visibility of overlap.
    ax_r.bar([N_STACK_X], [zp], width=N_STACK_WIDTH, bottom=zn, color=COL_POS_HI, edgecolor="black", linewidth=0.9, zorder=2)
    ax_r.bar([N_STACK_X], [zn], width=N_STACK_WIDTH, bottom=0.0, color=COL_NEG_LO, edgecolor="black", linewidth=0.9, zorder=4, alpha=0.62)
    _style_bar_axis(ax_r, stacked=True)
    if stage > 0:
        _draw_neg_stack_labels(ax_r, stage)


def _with_abs_stack(ax_r, stage, ylabel=None):
    h1, h2 = abs(ZAN), ZB4
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=N_STACK_WIDTH, bottom=0.0, color=COL_POS_LO, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=N_STACK_WIDTH, bottom=h1, color=COL_POS_HI, edgecolor="black", linewidth=0.9)
    _style_bar_axis(ax_r, stacked=True, ylabel=ylabel)
    fs = NORM_STACK_LABEL_FS
    zz = NORM_STACK_LABEL_Z
    if stage == 1:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$1/(1+4)$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$4/(1+4)$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 2:
        ax_r.text(N_STACK_X, 0.5 * h1, r"$\frac{1}{5}$", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            r"$\frac{4}{5}$",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )
    elif stage == 3:
        ax_r.text(N_STACK_X, 0.5 * h1, "20%", ha="center", va="center", fontsize=fs, fontweight="bold", color="#0a3d0a", zorder=zz)
        ax_r.text(
            N_STACK_X,
            h1 + 0.5 * h2,
            "80%",
            ha="center",
            va="center",
            fontsize=fs,
            fontweight="bold",
            color="#ffffff",
            path_effects=NORM_STACK_WHITE_PE,
            zorder=zz,
        )


_, SB4, EB4, ZB4, YB4 = _pick_realistic(6, 2)
_i_pos = _norm6b_pick_index_for_z(1.0, 1, SB4, EB4)
SA1, EA1, ZA1, YA1 = float(study_real[_i_pos]), float(exam_real[_i_pos]), float(z_real[_i_pos]), int(y_real[_i_pos])
_i_neg = _norm6b_pick_index_for_z(-1.0, 0, SB4, EB4)
SAN, EAN, ZAN, YAN = float(study_real[_i_neg]), float(exam_real[_i_neg]), float(z_real[_i_neg]), int(y_real[_i_neg])
assert np.isclose(ZA1, 1) and np.isclose(ZB4, 4) and np.isclose(ZAN, -1)
assert YA1 == 1 and YB4 == 1 and YAN == 0

# --- 31 ---
fig, (axl, axr) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
_draw_left_progress(axl, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
_draw_twin_bars(axr, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=2)
fig.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
save_fig(fig, "39_norm_positive_two_bars.png")

# --- 32 GIF ---
frames_nb = []
fig0, (ax0_l, ax0_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
fig0.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for _ in range(4):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=0)
    frames_nb.append(_norm_frame_png(fig0))
for _ in range(5):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=1)
    frames_nb.append(_norm_frame_png(fig0))
for _ in range(5):
    _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax0_r, ZA1, ZB4, COL_POS_LO, COL_POS_HI, phase=2)
    frames_nb.append(_norm_frame_png(fig0))
for st in (0, 1, 2, 3):
    nrep = 6 if st == 3 else 5
    for _ in range(nrep):
        _draw_left_progress(ax0_l, SA1, EA1, YA1, COL_POS_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_pos_stack(ax0_r, st)
        frames_nb.append(_norm_frame_png(fig0))
plt.close(fig0)
save_gif(frames_nb, "40_norm_positive_stack.gif", duration=NORM_MS)

# --- 33 GIF ---
frames_neg = []
f1, (ax1_l, ax1_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
f1.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for _ in range(4):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=0)
    frames_neg.append(_norm_frame_png(f1))
for _ in range(5):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=1, alpha_a=0.62)
    frames_neg.append(_norm_frame_png(f1))
for _ in range(5):
    _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax1_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=2, alpha_a=0.62)
    frames_neg.append(_norm_frame_png(f1))
for st in (0, 1, 2):
    nrep = 7 if st == 2 else 5
    for _ in range(nrep):
        _draw_left_progress(ax1_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_neg_stack(ax1_r, st)
        frames_neg.append(_norm_frame_png(f1))
plt.close(f1)
save_gif(frames_neg, "41_norm_negative_invalid.gif", duration=NORM_MS)

# --- 33b–33c: extra plane-only GIFs (same canvas as other dataset views) ---
def _frame_plane_2h_segment(show_segment: bool):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    fig.subplots_adjust(**EXPORT_ADJ)
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label="ST - EL = 0",
        color="black",
        linewidth=1,
        x_range=tuple(ax.get_xlim()),
    )
    if show_segment:
        ax.plot([6, 4], [5, 5], color="#c44e00", linewidth=3.2, solid_capstyle="round", zorder=5)
        ax.text(
            5.0,
            5.48,
            "2 h difference",
            ha="center",
            va="bottom",
            fontsize=NOTE_SIZE + 2,
            fontweight="bold",
            color="#5c2100",
        )
    add_combined_legend(ax, loc="upper left")
    return fig_to_image(fig, tight_layout=False)


frames_41b = []
for _ in range(8):
    frames_41b.append(_frame_plane_2h_segment(False))
for _ in range(22):
    frames_41b.append(_frame_plane_2h_segment(True))
for _ in range(14):
    frames_41b.append(_frame_plane_2h_segment(True))
save_gif(frames_41b, "41b_dataset_plane_2h_difference_segment.gif", duration=NORM_MS)


def _frame_plane_zero_study_contrast(step: int):
    """step 0 base; 1 vertical same-ST; 2 add (0,1) perp; 3 add (0,4) perp; 4 caption."""
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    fig.subplots_adjust(**EXPORT_ADJ)
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_threshold_line(
        ax,
        shift=midpoint_shift,
        label="ST - EL = 0",
        color="black",
        linewidth=1,
        x_range=tuple(ax.get_xlim()),
    )
    if step >= 1:
        ax.plot([0, 0], [1, 4], color="#4a4a4a", linewidth=2.9, solid_capstyle="round", zorder=4)
        ax.text(
            0.42,
            2.5,
            "0 h study\n(1 h vs 4 h exam)",
            ha="left",
            va="center",
            fontsize=NOTE_SIZE,
            color="#2a2a2a",
        )
    if step >= 2:
        _draw_student_distance(ax, 0.0, 1.0, 0, "#2c7fb8")
    if step >= 3:
        _draw_student_distance(ax, 0.0, 4.0, 0, "#e34a33")
    if step >= 4:
        ax.text(
            1.05,
            0.35,
            "Longer exam with no study\n→ farther below ST−EL = 0\n(less work vs the bar)",
            ha="left",
            va="bottom",
            fontsize=NOTE_SIZE,
            color="#141414",
            bbox=dict(boxstyle="round,pad=0.26", facecolor="white", edgecolor="#bbbbbb", linewidth=0.65, alpha=0.95),
        )
    add_combined_legend(ax, loc="upper left")
    return fig_to_image(fig, tight_layout=False)


frames_41c = []
for _ in range(6):
    frames_41c.append(_frame_plane_zero_study_contrast(0))
for _ in range(10):
    frames_41c.append(_frame_plane_zero_study_contrast(1))
for _ in range(8):
    frames_41c.append(_frame_plane_zero_study_contrast(2))
for _ in range(8):
    frames_41c.append(_frame_plane_zero_study_contrast(3))
for _ in range(14):
    frames_41c.append(_frame_plane_zero_study_contrast(4))
save_gif(frames_41c, "41c_dataset_plane_zero_study_exam_gap_contrast.gif", duration=NORM_MS)


# --- 34 GIF ---
frames_abs = []
f2, (ax2_l, ax2_r) = plt.subplots(1, 2, figsize=NORM_DUO_FIGSIZE, gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28})
f2.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
# signed reveal: none -> A -> A+B
for _ in range(4):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=0)
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=1, alpha_a=0.62)
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax2_r, ZAN, ZB4, COL_NEG_LO, COL_POS_HI, phase=2, alpha_a=0.62)
    frames_abs.append(_norm_frame_png(f2))
# absolute twin reveal: none -> A -> A+B
for _ in range(4):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=0)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=0, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=1)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=1, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for _ in range(5):
    _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
    _draw_twin_bars(ax2_r, abs(ZAN), ZB4, COL_POS_LO, COL_POS_HI, phase=2, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
    frames_abs.append(_norm_frame_png(f2))
for pst in (1, 2, 3):
    nrep = 7 if pst == 3 else 5
    for _ in range(nrep):
        _draw_left_progress(ax2_l, SAN, EAN, YAN, COL_NEG_LO, SB4, EB4, YB4, COL_POS_HI, phase=2)
        _with_abs_stack(ax2_r, pst, ylabel=r"$|\mathrm{ST}-\mathrm{EL}|$")
        frames_abs.append(_norm_frame_png(f2))
plt.close(f2)
save_gif(frames_abs, "42_norm_abs_pitfall.gif", duration=NORM_MS)

# --- 35: GIF — |z| bars (same cadence as 36); negative z shows signed bar then flips to |z|
ZS_BAR_HINT = np.arange(-3, 4, dtype=float)
BARHINT_XLIM = (-3.9, 3.9)
BARHINT_BAR_W = 0.34
ABS_F_YLIM = (-3.9, 3.9)
ABS_F_YTICKS = np.arange(-3, 5)
_ABS_F_YLABEL = r"$|\mathrm{ST}-\mathrm{EL}|$"
BARHINT_NONZERO_ORDER = (-3, -2, -1, 1, 2, 3)


def _build_barhint_points(zs_vals):
    sh = float(midpoint_shift)
    want = {int(z) for z in zs_vals}
    chosen_s = [4.0]
    chosen_e = [4.0]
    out = {0: (4.0, 4.0, 1)}
    for zt in BARHINT_NONZERO_ORDER:
        if zt not in want:
            continue
        want_y = 1 if zt > 0 else 0
        idxs = np.flatnonzero(np.isclose(z_real, float(zt), atol=1e-9) & (y_real == want_y))
        if not idxs.size:
            idxs = np.flatnonzero(np.isclose(z_real, float(zt), atol=1e-9))
        best_i = None
        best_key = None
        relax_i = None
        relax_key = None
        for i in idxs:
            s = float(study_real[i])
            ee = float(exam_real[i])
            y = int(y_real[i])
            ok = True
            for sj, ej in zip(chosen_s, chosen_e):
                if _norm6b_on_perp_open(sj, ej, s, ee, sh) or _norm6b_on_perp_open(s, ee, sj, ej, sh):
                    ok = False
                    break
            md = min(np.hypot(s - sj, ee - ej) for sj, ej in zip(chosen_s, chosen_e))
            key = (md, -s)
            if relax_i is None or key > relax_key:
                relax_key, relax_i = key, int(i)
            if ok and (best_i is None or key > best_key):
                best_key, best_i = key, int(i)
        use_i = int(best_i if best_i is not None else relax_i)
        s = float(study_real[use_i])
        ee = float(exam_real[use_i])
        y = int(y_real[use_i])
        out[int(zt)] = (s, ee, y)
        chosen_s.append(s)
        chosen_e.append(ee)
    return out


BARHINT_POINTS = _build_barhint_points(ZS_BAR_HINT)


def _style_abs_f_axis(ax):
    ax.set_xlim(*BARHINT_XLIM)
    ax.set_xticks(np.arange(-3, 4, dtype=int))
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylim(*ABS_F_YLIM)
    ax.set_yticks(ABS_F_YTICKS)
    ax.set_ylabel(_ABS_F_YLABEL, fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")


def _draw_abs_f_bars(ax_r, specs):
    ax_r.clear()
    for z, mode in specs:
        col = _exp_bar_facecolors([float(z)])[0]
        if mode == "signed" and z < 0:
            ax_r.bar([z], [z], width=BARHINT_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
        else:
            h = float(abs(z))
            ax_r.bar([z], [h], width=BARHINT_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
    _style_abs_f_axis(ax_r)


def _draw_barhint_left(ax_l, zs_show, points_map):
    _draw_plane_base(ax_l)
    for z in zs_show:
        zi = int(z)
        s, e, y = points_map[zi]
        col = _exp_bar_facecolors([float(zi)])[0]
        _draw_barhint_plane_marker(ax_l, s, e, y, float(zi), col)


frames_abs_f = []
fig_af, (ax_af_l, ax_af_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_af.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for k in range(len(ZS_BAR_HINT)):
    z_new = float(ZS_BAR_HINT[k])
    if z_new < 0:
        for _ in range(4):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k)] + [(z_new, "signed")]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
        for _ in range(4):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k + 1)]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
    else:
        for _ in range(5):
            zs_show = list(ZS_BAR_HINT[: k + 1])
            specs = [(float(ZS_BAR_HINT[j]), "exp") for j in range(k + 1)]
            _draw_barhint_left(ax_af_l, zs_show, BARHINT_POINTS)
            _draw_abs_f_bars(ax_af_r, specs)
            frames_abs_f.append(_norm_frame_png(fig_af))
plt.close(fig_af)
save_gif(frames_abs_f, "43_norm_abs_bar_hint.gif", duration=NORM_MS)

# --- 36: GIF — 2^z with dataset + perpendicular distances; negative z shows signed bar then flips to f(z)=2^z
ZS_EXP = ZS_BAR_HINT
EXP_XLIM = BARHINT_XLIM
EXP_YLIM = (-3.9, 9.5)
EXP_YTICKS = np.arange(-3, 10)
EXP_BAR_W = BARHINT_BAR_W
_EXP_YLABEL = r"$f(\mathrm{ST}-\mathrm{EL})$"
EXP_POINTS = BARHINT_POINTS
# 36 right panel: effort gloss near (-3, 3); exponential-ease arrow (-2, 3) -> (2, 8)
EXP36_EFFORT_XY = (-3.0, 3.0)
EXP36_ARROW_A = (-2.0, 3.0)
EXP36_ARROW_B = (2.0, 8.0)
EXP36_ARROW_K = 2.8  # larger => tighter bend toward the end (exponential feel)

def _style_exp_f_axis(ax):
    ax.set_xlim(*EXP_XLIM)
    ax.set_xticks(np.arange(-3, 4, dtype=int))
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylim(*EXP_YLIM)
    ax.set_yticks(EXP_YTICKS)
    ax.set_ylabel(_EXP_YLABEL, fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")


def _draw_exp_f_bars(ax_r, specs, arrow_progress=1.0):
    ax_r.clear()
    for z, mode in specs:
        col = _exp_bar_facecolors([float(z)])[0]
        if mode == "signed" and z < 0:
            ax_r.bar([z], [z], width=EXP_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
        else:
            h = float(np.power(2.0, z))
            ax_r.bar([z], [h], width=EXP_BAR_W, bottom=0.0, color=col, edgecolor="black", linewidth=0.85, zorder=3)
    _style_exp_f_axis(ax_r)
    eff_bbox = dict(
        boxstyle="round,pad=0.22",
        facecolor="white",
        edgecolor="#bbbbbb",
        linewidth=0.65,
        alpha=0.95,
    )
    ax_r.text(
        EXP36_EFFORT_XY[0],
        EXP36_EFFORT_XY[1],
        r"$\mathrm{effort}$",
        fontsize=NORM_WORK_EFFORT_GLOSS_Z,
        ha="left",
        va="center",
        bbox=eff_bbox,
        zorder=25,
    )
    if arrow_progress > 1e-9:
        ta = np.linspace(0.0, 1.0, 96)
        k = EXP36_ARROW_K
        xa = EXP36_ARROW_A[0] + (EXP36_ARROW_B[0] - EXP36_ARROW_A[0]) * ta
        ya = (
            EXP36_ARROW_A[1]
            + (EXP36_ARROW_B[1] - EXP36_ARROW_A[1])
            * (np.exp(k * ta) - 1.0)
            / (np.exp(k) - 1.0)
        )
        verts = np.column_stack([xa, ya])
        pr = float(np.clip(arrow_progress, 0.0, 1.0))
        n = len(verts)
        end = max(2, int(np.ceil(pr * (n - 1))) + 1)
        sub = verts[: min(end, n)]
        codes = [MplPath.MOVETO] + [MplPath.LINETO] * (len(sub) - 1)
        ax_r.add_patch(
            FancyArrowPatch(
                path=MplPath(sub, codes),
                arrowstyle="-|>",
                mutation_scale=14,
                linewidth=1.35,
                facecolor="#2d2d2d",
                edgecolor="#2d2d2d",
                zorder=24,
            )
        )


def _draw_exp_left(ax_l, zs_show):
    _draw_plane_base(ax_l)
    for z in zs_show:
        s, e, y = EXP_POINTS[int(z)]
        col = _exp_bar_facecolors([float(z)])[0]
        _draw_student_distance(ax_l, s, e, y, col)


frames_eb = []
_EXP36_TOT_FR = sum(8 if float(ZS_EXP[k]) < 0 else 5 for k in range(len(ZS_EXP)))
_eb_arrow_i = 0
fig_e, (ax_e_l, ax_e_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_e.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for k in range(len(ZS_EXP)):
    z_new = float(ZS_EXP[k])
    if z_new < 0:
        for _ in range(4):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k)] + [(z_new, "signed")]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(
                ax_e_r,
                specs,
                arrow_progress=_eb_arrow_i / max(_EXP36_TOT_FR - 1, 1),
            )
            _eb_arrow_i += 1
            frames_eb.append(_norm_frame_png(fig_e))
        for _ in range(4):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k + 1)]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(
                ax_e_r,
                specs,
                arrow_progress=_eb_arrow_i / max(_EXP36_TOT_FR - 1, 1),
            )
            _eb_arrow_i += 1
            frames_eb.append(_norm_frame_png(fig_e))
    else:
        for _ in range(5):
            zs_show = list(ZS_EXP[: k + 1])
            specs = [(float(ZS_EXP[j]), "exp") for j in range(k + 1)]
            _draw_exp_left(ax_e_l, zs_show)
            _draw_exp_f_bars(
                ax_e_r,
                specs,
                arrow_progress=_eb_arrow_i / max(_EXP36_TOT_FR - 1, 1),
            )
            _eb_arrow_i += 1
            frames_eb.append(_norm_frame_png(fig_e))
plt.close(fig_e)
save_gif(frames_eb, "44_norm_exp_bar_hint.gif", duration=NORM_MS)

# --- 41b: GIF — signed hold → convert each bar to exp in list order (-3…3) → effort label →
# emphasize each e-bar (smallest exp height first); one arrow grows smoothly from first→last bar across the whole pass.
HOLD_41B_STEL = 6
HOLD_41B_CONV = 4  # frames per bar when flipping that bar to exp
HOLD_41B_EFFORT_APPEAR = 6  # all exp, show effort text, no arrow yet
HOLD_41B_EMPH = 6  # hold frames per bar during emphasis; arrow advances incrementally across all bars
LW_41B_NORM = 0.85
LW_41B_BOLD = 3.6
ZS_41B_ALL = [float(z) for z in ZS_BAR_HINT]
ZS_41B_BY_EXP = sorted(ZS_41B_ALL, key=lambda z: float(np.exp(z)))
# 41b-only arrow: one smooth path from smallest-exp bar top → largest-exp bar top.
_z41b_lo = float(ZS_41B_BY_EXP[0])
_z41b_hi = float(ZS_41B_BY_EXP[-1])
EXP41B_ARROW_A = (_z41b_lo, float(np.exp(_z41b_lo)))
EXP41B_ARROW_B = (_z41b_hi, float(np.exp(_z41b_hi)))
EXP41B_ARROW_K = 2.85  # ease along chord (smaller => gentler bend than old fixed A/B)
EXP41B_ARROW_N = 120


def _style_signed_stel_multi_axis(ax_r):
    ax_r.set_xlim(*BARHINT_XLIM)
    ax_r.set_xticks(np.arange(-3, 4, dtype=int))
    ax_r.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax_r.set_ylim(-4.0, 22.0)
    ax_r.set_yticks(np.arange(-4, 23, 2, dtype=int))
    ax_r.set_ylabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax_r.tick_params(axis="both", labelsize=FONT_SIZE)
    ax_r.grid(axis="y", alpha=0.22)
    ax_r.axhline(0.0, color="black", linewidth=1.25, linestyle="-", zorder=1)


def _draw_41b_signed_stel_bars(ax_r, emph_z=None):
    ax_r.clear()
    for zi in ZS_41B_ALL:
        col = _exp_bar_facecolors([zi])[0]
        lw = LW_41B_BOLD if emph_z is not None and abs(zi - float(emph_z)) < 1e-9 else LW_41B_NORM
        ax_r.bar(
            [zi],
            [zi],
            width=BARHINT_BAR_W,
            bottom=0.0,
            color=col,
            edgecolor="black",
            linewidth=lw,
            zorder=3,
        )
    _style_signed_stel_multi_axis(ax_r)


def _style_exp_e_multi_axis(ax_r):
    ax_r.set_xlim(*BARHINT_XLIM)
    ax_r.set_xticks(np.arange(-3, 4, dtype=int))
    ax_r.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax_r.set_ylim(-4.0, 22.0)
    ax_r.set_yticks(np.arange(-4, 23, 2, dtype=int))
    ax_r.set_ylabel(r"$e^{\mathrm{ST}-\mathrm{EL}}$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax_r.tick_params(axis="both", labelsize=FONT_SIZE)
    ax_r.grid(axis="y", alpha=0.22)
    ax_r.axhline(0.0, color="black", linewidth=1.25, linestyle="-", zorder=1)


def _draw_41b_hybrid_bars(
    ax_r,
    n_exp_bars,
    *,
    emph_z=None,
    show_effort=False,
    arrow_progress=0.0,
):
    """First `n_exp_bars` z-slots (in `ZS_41B_ALL` order) use exp height; the rest stay signed (height = z)."""
    ax_r.clear()
    heights_exp = {float(z): float(np.exp(float(z))) for z in ZS_41B_ALL}
    for i, zi in enumerate(ZS_41B_ALL):
        h = heights_exp[zi] if i < int(n_exp_bars) else float(zi)
        col = _exp_bar_facecolors([zi])[0]
        lw = LW_41B_BOLD if emph_z is not None and abs(zi - float(emph_z)) < 1e-9 else LW_41B_NORM
        ax_r.bar(
            [zi],
            [h],
            width=BARHINT_BAR_W,
            bottom=0.0,
            color=col,
            edgecolor="black",
            linewidth=lw,
            zorder=3,
        )
    _style_exp_e_multi_axis(ax_r)
    if show_effort:
        eff_bbox = dict(
            boxstyle="round,pad=0.22",
            facecolor="white",
            edgecolor="#bbbbbb",
            linewidth=0.65,
            alpha=0.95,
        )
        ax_r.text(
            EXP36_EFFORT_XY[0],
            EXP36_EFFORT_XY[1],
            r"$\mathrm{effort}$",
            fontsize=NORM_WORK_EFFORT_GLOSS_Z,
            ha="left",
            va="center",
            bbox=eff_bbox,
            zorder=25,
        )
    if arrow_progress > 1e-9:
        ta = np.linspace(0.0, 1.0, EXP41B_ARROW_N)
        k = float(EXP41B_ARROW_K)
        xa = EXP41B_ARROW_A[0] + (EXP41B_ARROW_B[0] - EXP41B_ARROW_A[0]) * ta
        ya = (
            EXP41B_ARROW_A[1]
            + (EXP41B_ARROW_B[1] - EXP41B_ARROW_A[1])
            * (np.exp(k * ta) - 1.0)
            / (np.exp(k) - 1.0)
        )
        verts = np.column_stack([xa, ya])
        pr = float(np.clip(arrow_progress, 0.0, 1.0))
        n = len(verts)
        end = max(2, int(np.ceil(pr * (n - 1))) + 1)
        sub = verts[: min(end, n)]
        codes = [MplPath.MOVETO] + [MplPath.LINETO] * (len(sub) - 1)
        ax_r.add_patch(
            FancyArrowPatch(
                path=MplPath(sub, codes),
                arrowstyle="-|>",
                mutation_scale=14,
                linewidth=1.35,
                facecolor="#2d2d2d",
                edgecolor="#2d2d2d",
                zorder=24,
            )
        )


def _draw_41b_exp_e_bars(ax_r, emph_z=None, arrow_progress=1.0, show_effort=True):
    _draw_41b_hybrid_bars(
        ax_r,
        len(ZS_41B_ALL),
        emph_z=emph_z,
        show_effort=show_effort,
        arrow_progress=arrow_progress,
    )


frames_41b = []
fig_41b, (ax_41b_l, ax_41b_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_41b.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
zs_all_show = [int(z) for z in ZS_41B_ALL]
for _ in range(HOLD_41B_STEL):
    _draw_barhint_left(ax_41b_l, zs_all_show, BARHINT_POINTS)
    _draw_41b_signed_stel_bars(ax_41b_r, emph_z=None)
    _norm_work_effort_gloss(ax_41b_r, "work")
    frames_41b.append(_norm_frame_png(fig_41b))
for nb in range(1, len(ZS_41B_ALL) + 1):
    for _ in range(HOLD_41B_CONV):
        _draw_barhint_left(ax_41b_l, zs_all_show, BARHINT_POINTS)
        _draw_41b_hybrid_bars(
            ax_41b_r,
            nb,
            emph_z=None,
            show_effort=False,
            arrow_progress=0.0,
        )
        _norm_work_effort_gloss(ax_41b_r, "work")
        frames_41b.append(_norm_frame_png(fig_41b))
for _ in range(HOLD_41B_EFFORT_APPEAR):
    _draw_barhint_left(ax_41b_l, zs_all_show, BARHINT_POINTS)
    _draw_41b_hybrid_bars(
        ax_41b_r,
        len(ZS_41B_ALL),
        emph_z=None,
        show_effort=True,
        arrow_progress=0.0,
    )
    _norm_work_effort_gloss(ax_41b_r, "work")
    frames_41b.append(_norm_frame_png(fig_41b))
_41b_emph_tot_fr = len(ZS_41B_BY_EXP) * HOLD_41B_EMPH
_41b_emph_i = 0
for emph in ZS_41B_BY_EXP:
    for _ in range(HOLD_41B_EMPH):
        pr = (_41b_emph_i + 1) / float(max(_41b_emph_tot_fr, 1))
        _draw_barhint_left(ax_41b_l, zs_all_show, BARHINT_POINTS)
        _draw_41b_hybrid_bars(
            ax_41b_r,
            len(ZS_41B_ALL),
            emph_z=emph,
            show_effort=True,
            arrow_progress=pr,
        )
        _norm_work_effort_gloss(ax_41b_r, "work")
        frames_41b.append(_norm_frame_png(fig_41b))
        _41b_emph_i += 1
plt.close(fig_41b)
save_gif(frames_41b, "44b_norm_all_st_el_then_exp_emphasis.gif", duration=NORM_MS)


## Scene 7 - Exponential step-by-step ($2^x$, step size 1 in $x$)

Script alignment (Chapter 1 script):
- moving **one unit left** in $x$ (so $\Delta x=-1$) must shrink the positive output while leaving room for all negatives
- the consistent rule is **multiply by $\frac{1}{2}$** each step — that is exactly **$2^x$** on integer steps

- **Scene 6b (norm duo, exports ~37–42)** motivates exponentials: **`43_norm_abs_bar_hint.gif`/`44_norm_exp_bar_hint.gif`** step **$z\in[-3,3]$** on the bars (**$z=0$** on the plane is a **black circle**); the abs stack uses **$|z|$** heights; the exp stack uses **$2^{z}$** (strictly positive, increasing in $z$), same **red–white–green** colormap as sigmoid **3D**

`45_exponential_point_guides.gif` introduces a single point with dotted coordinate guides: first the **vertical** guide to its $x$ tick, then the **horizontal** guide to its $y$ tick.

Exports **44**–**51** use axis labels **WORK** ($x$) and **EFFORT** ($y$). `45b_exponential_effort_unit_brace.gif` opens like **44** (curve + point only), then **bold**-draws the **x**-axis from the plot’s left edge to **$x=0$**, then a **bold** vertical from **$(0,0)$** to the point.

`46_exponential_mapping.gif` is the local $2^x$ panel: **dotted** blue curve (**linewidth 1**), **blue** point marker; each step shows **vertical segment first**, then its label, then the next horizontal segment (label cleared).

`47_step_left_drop_one_to_zero.png` snapshots one step left with a full output drop of **1** to zero. `48_step_two_drop_half_to_zero.png` snapshots the second step with a drop of **1/2** (instead of **1/4**) to zero.

`49_exponential_transition.gif` continues from the **final frame of `46_exponential_mapping.gif`**: stairs vanish, the **$2^x$** trace becomes **solid** (**lw 2**), then the window **opens** to $x\in(-4.5,6)$ (with matching $y$-scale). Then **`50_exponential_2x_legend.png` / `51_exponential_ex_legend.png`** are static **$(-4.5,6)$** plots with legends $f(x)=2^x$ and $f(x)=e^x$ (same blue curve color). **`53_norm_exp_positive_softmax.gif`** / **`54_norm_exp_negative_softmax.gif`** replay the **`40_norm_positive_stack.gif`** / **`41_norm_negative_invalid.gif`** duo-panel cadence with **$e^{\mathrm{ST}-\mathrm{EL}}$** bars, a stacked **softmax** denominator, explicit normalization fractions, then **three-decimal** masses. **`55_far_study_hours_zoom.gif`** opens on the **duo** layout: the **right** panel is an empty **ST − EL** bar axis until the **left** finishes zooming to study time $[4,120]$ and exam length $[4,110]$ with **ST $-$ EL** gaps **101** (student **A**) and **104** (student **B**), and **both** are **selected** (perpendicular guides); then **twin ST − EL** bars, **exp** bars, stacked **softmax**, and the same probability callout as **`53_norm_exp_positive_softmax.gif`**/**`54_norm_exp_negative_softmax.gif`**. **`56_softmax_threshold_and_z1.gif`** places **B** on **ST − EL = 0** and **A** at **ST − EL = 1** with the same **exp**/**softmax** beats. Scene **8** then adds **four** softmax snapshot PNGs (**`57_softmax_pf_stack_normalized_frac.png`–`60_softmax_pf_stack_fail_eneg1.png`**), **two** **2D** sigmoid reveal GIFs (**`62_sigmoid_2d_normalized_exp_reveal.gif`**, **`66_sigmoid_sigma_negz_reveal.gif`**), and **three** matching **2D** sigmoid legend panels (**`67_sigmoid_2d_normalized_exp_legend.png`–`69_sigmoid_2d_standard_legend.png`**) before the **3D** rotation exports.


In [56]:
# Scene 7 exponential — shared setup (helpers + constants for exports 42–58).
EXP_STEP_HOLD = 5
STAIR_LW = 1
CURVE_COLOR = "#1f77b4"
CURVE_LW = 1.0

frames = []


def _strip(ax):
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    ax.tick_params(labelbottom=True, labelleft=True, labelsize=FONT_SIZE)


def _strip_work_effort(ax):
    ax.set_title("")
    ax.tick_params(labelbottom=True, labelleft=True, labelsize=FONT_SIZE)
    ax.set_xlabel("WORK", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("EFFORT", fontsize=AXIS_LABEL_SIZE, labelpad=10)


def _v_step_label(s):
    return {1: "1/2", 2: "1/4", 3: "1/8"}.get(s, "...")


def _plot_curve(ax, xs, x_min):
    m = xs >= x_min
    ax.plot(
        xs[m],
        np.power(2.0, xs[m]),
        color=CURVE_COLOR,
        linestyle=":",
        linewidth=CURVE_LW,
        zorder=2,
    )


def _plot_full_stairs(ax, upto_s_inclusive, label_vertical_s=None):
    for s in range(1, upto_s_inclusive + 1):
        xp = -(s - 1)
        xc = -s
        yp = float(np.power(2.0, xp))
        yc = float(np.power(2.0, xc))
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        if label_vertical_s is not None and s == label_vertical_s:
            vy = 0.5 * (yp + yc)
            ax.text(
                xc - 0.12,
                vy,
                _v_step_label(s),
                fontsize=ANNOT_SIZE + 2,
                ha="right",
                va="center",
                color="black",
                zorder=5,
            )


def _plot_horizontal_only(ax, s):
    xp = -(s - 1)
    xc = -s
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xp, xc],
        [yp, yp],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )
    return xp, xc, yp, yc


def _stair_h(ax, k):
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xp, xc],
        [yp, yp],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )
    return xp, xc, yp, yc


def _stair_v(ax, k):
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.plot(
        [xc, xc],
        [yp, yc],
        color="black",
        linewidth=STAIR_LW,
        solid_capstyle="projecting",
        zorder=4,
    )


xs_full = np.linspace(-4.5, 1.4, 520)


def _exp_snap(fig, ax):
    finalize_exponential_figure(fig)
    return fig_to_image(fig, tight_layout=False)


def _save_exp_png(fig, filename):
    fig.savefig(
        OUTPUT_DIR / filename,
        format="png",
        dpi=EXPORT_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


# --- 37: point-only coordinate guide (vertical then horizontal dotted helpers) ---
COORD_X = 0.0
COORD_Y = 1.0
frames_coord = []
for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=162,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames_coord.append(_exp_snap(fig, ax))

for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.plot([COORD_X, COORD_X], [0.0, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=162,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames_coord.append(_exp_snap(fig, ax))

for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, COORD_X)
    ax.plot([COORD_X, COORD_X], [0.0, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.plot([-4.5, COORD_X], [COORD_Y, COORD_Y], color="black", linestyle=":", linewidth=1.2, zorder=4)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=162,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames_coord.append(_exp_snap(fig, ax))

save_gif(frames_coord, "45_exponential_point_guides.gif", duration=70)


# --- 44b: like 44 start → bold x-axis from left to x=0 → bold vertical (0,0) to point.
BRACE_MS = 70
X_LEFT_44B = -4.5
N44B_HOLD = EXP_STEP_HOLD
N44B_H_FR = 18
N44B_V_FR = 14
BW44B_AXIS = 3.8
BW44B_VERT = 3.8
frames_42b = []


def _frame_44b(ax, *, horiz_t=0.0, vert_t=0.0):
    """horiz_t in [0,1]: bold x-axis segment on y=0 from x_left toward 0; vert_t in [0,1]: bold (0,0)→(0,COORD_Y)."""
    _plot_curve(ax, xs_full, COORD_X)
    ax.scatter(
        [COORD_X],
        [COORD_Y],
        s=162,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    if horiz_t > 1e-9:
        x2 = X_LEFT_44B + float(horiz_t) * (0.0 - X_LEFT_44B)
        ax.plot(
            [X_LEFT_44B, x2],
            [0.0, 0.0],
            color="black",
            linewidth=BW44B_AXIS,
            solid_capstyle="butt",
            zorder=9,
        )
    if vert_t > 1e-9:
        y2 = float(vert_t) * COORD_Y
        ax.plot(
            [0.0, 0.0],
            [0.0, y2],
            color="black",
            linewidth=BW44B_VERT,
            solid_capstyle="projecting",
            zorder=10,
        )


for _ in range(N44B_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_44b(ax, horiz_t=0.0, vert_t=0.0)
    frames_42b.append(_exp_snap(fig, ax))
for hi in range(1, N44B_H_FR + 1):
    t = hi / float(N44B_H_FR)
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_44b(ax, horiz_t=t, vert_t=0.0)
    frames_42b.append(_exp_snap(fig, ax))
for _ in range(N44B_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_44b(ax, horiz_t=1.0, vert_t=0.0)
    frames_42b.append(_exp_snap(fig, ax))
for vi in range(1, N44B_V_FR + 1):
    t = vi / float(N44B_V_FR)
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_44b(ax, horiz_t=1.0, vert_t=t)
    frames_42b.append(_exp_snap(fig, ax))
for _ in range(N44B_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_44b(ax, horiz_t=1.0, vert_t=1.0)
    frames_42b.append(_exp_snap(fig, ax))

save_gif(frames_42b, "45b_exponential_effort_unit_brace.gif", duration=BRACE_MS)



for _ in range(EXP_STEP_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, 0.0)
    ax.scatter(
        [0.0],
        [1.0],
        s=162,
        facecolors=CURVE_COLOR,
        edgecolors="white",
        linewidths=0.9,
        zorder=6,
        clip_on=False,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames.append(_exp_snap(fig, ax))

for s in range(1, 7):
    xp = -(s - 1)
    yp = float(np.power(2.0, xp))
    xc = -s
    yc = float(np.power(2.0, xc))

    # 1) move across: horizontal for step s (completed through vertical s-1)
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xp)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.scatter(
            [xc],
            [yp],
            s=162,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip_work_effort(ax)
        frames.append(_exp_snap(fig, ax))

    # 2) move down: vertical for step s
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xc)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.scatter(
            [xc],
            [yc],
            s=162,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip_work_effort(ax)
        frames.append(_exp_snap(fig, ax))

    # 3) label for this vertical drop
    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
        _plot_curve(ax, xs_full, xc)
        for k in range(1, s):
            _stair_h(ax, k)
            _stair_v(ax, k)
        ax.plot(
            [xp, xc],
            [yp, yp],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.plot(
            [xc, xc],
            [yp, yc],
            color="black",
            linewidth=STAIR_LW,
            solid_capstyle="projecting",
            zorder=4,
        )
        ax.text(
            xc - 0.12,
            0.5 * (yp + yc),
            _v_step_label(s),
            fontsize=ANNOT_SIZE + 2,
            ha="right",
            va="center",
            color="black",
            zorder=5,
        )
        ax.scatter(
            [xc],
            [yc],
            s=162,
            facecolors=CURVE_COLOR,
            edgecolors="white",
            linewidths=0.9,
            zorder=6,
            clip_on=False,
        )
        ax.set_xlim(-4.5, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip_work_effort(ax)
        frames.append(_exp_snap(fig, ax))

    # 4) move across (next horizontal) + remove label
    if s < 6:
        xp2 = -s
        xc2 = -(s + 1)
        yp2 = float(np.power(2.0, xp2))
        for _ in range(EXP_STEP_HOLD):
            fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
            _plot_curve(ax, xs_full, xp2)
            for k in range(1, s + 1):
                _stair_h(ax, k)
                _stair_v(ax, k)
            ax.plot(
                [xp2, xc2],
                [yp2, yp2],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.scatter(
                [xc2],
                [yp2],
                s=162,
                facecolors=CURVE_COLOR,
                edgecolors="white",
                linewidths=0.9,
                zorder=6,
                clip_on=False,
            )
            ax.set_xlim(-4.5, 1.4)
            ax.set_ylim(0, 1.15)
            ax.grid(alpha=0.2)
            _strip_work_effort(ax)
            frames.append(_exp_snap(fig, ax))
    else:
        for _ in range(EXP_STEP_HOLD):
            fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
            _plot_curve(ax, xs_full, xc)
            for k in range(1, s):
                _stair_h(ax, k)
                _stair_v(ax, k)
            ax.plot(
                [xp, xc],
                [yp, yp],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.plot(
                [xc, xc],
                [yp, yc],
                color="black",
                linewidth=STAIR_LW,
                solid_capstyle="projecting",
                zorder=4,
            )
            ax.scatter(
                [xc],
                [yc],
                s=162,
                facecolors=CURVE_COLOR,
                edgecolors="white",
                linewidths=0.9,
                zorder=6,
                clip_on=False,
            )
            ax.set_xlim(-4.5, 1.4)
            ax.set_ylim(0, 1.15)
            ax.grid(alpha=0.2)
            _strip_work_effort(ax)
            frames.append(_exp_snap(fig, ax))

save_gif(frames, "46_exponential_mapping.gif", duration=70)

# --- 38: snapshot after one left step with a full drop of 1 to zero ---
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
_plot_curve(ax, xs_full, 0.0)
_plot_horizontal_only(ax, 1)
ax.plot(
    [-1.0, -1.0],
    [1.0, 0.0],
    color="black",
    linewidth=STAIR_LW,
    solid_capstyle="projecting",
    zorder=4,
)
ax.text(-1.12, 0.5, "1", fontsize=ANNOT_SIZE + 2, ha="right", va="center", color="black", zorder=5)
ax.scatter(
    [-1.0],
    [0.0],
    s=162,
    facecolors=CURVE_COLOR,
    edgecolors="white",
    linewidths=0.9,
    zorder=6,
    clip_on=False,
)
ax.set_xlim(-4.5, 1.4)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.2)
_strip_work_effort(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "47_step_left_drop_one_to_zero.png")

# --- 39: snapshot at second step with a drop of 1/2 (instead of 1/4) to zero ---
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
_plot_curve(ax, xs_full, -1.0)
_plot_full_stairs(ax, 1, label_vertical_s=None)
_plot_horizontal_only(ax, 2)
ax.plot(
    [-2.0, -2.0],
    [0.5, 0.0],
    color="black",
    linewidth=STAIR_LW,
    solid_capstyle="projecting",
    zorder=4,
)
ax.text(-2.12, 0.25, "1/2", fontsize=ANNOT_SIZE + 2, ha="right", va="center", color="black", zorder=5)
ax.scatter(
    [-2.0],
    [0.0],
    s=162,
    facecolors=CURVE_COLOR,
    edgecolors="white",
    linewidths=0.9,
    zorder=6,
    clip_on=False,
)
ax.set_xlim(-4.5, 1.4)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.2)
_strip_work_effort(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "48_step_two_drop_half_to_zero.png")

# --- 41: from end of 38 → remove stairs → solid curve → grow x-range to (-4.5, 6) ---
TRANS_MS = 55
HOLD_MATCH_25 = 5
HOLD_NO_STAIRS = 5
HOLD_SOLID_PREZOOM = 5
N_GROW = 40
xs_wide = np.linspace(-4.5, 6.0, 960)

frames26 = []


def _frame_end_of_25(ax):
    _plot_curve(ax, xs_full, -6.0)
    for k in range(1, 7):
        _stair_h(ax, k)
        _stair_v(ax, k)
    k = 6
    xp = -(k - 1)
    xc = -k
    yp = float(np.power(2.0, xp))
    yc = float(np.power(2.0, xc))
    ax.text(
        xc - 0.12,
        0.5 * (yp + yc),
        _v_step_label(k),
        fontsize=ANNOT_SIZE + 2,
        ha="right",
        va="center",
        color="black",
        zorder=5,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
for _ in range(HOLD_MATCH_25):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _frame_end_of_25(ax)
    frames26.append(_exp_snap(fig, ax))

for _ in range(HOLD_NO_STAIRS):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _plot_curve(ax, xs_full, -6.0)
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames26.append(_exp_snap(fig, ax))

for _ in range(HOLD_SOLID_PREZOOM):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    ax.plot(
        xs_wide,
        np.power(2.0, xs_wide),
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(-4.5, 1.4)
    ax.set_ylim(0, 1.15)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames26.append(_exp_snap(fig, ax))

for i in range(N_GROW):
    t = i / (N_GROW - 1) if N_GROW > 1 else 1.0
    x0 = -4.5
    x1 = 1.4 + t * (6.0 - 1.4)
    y1 = 2 ** (x1 - 1.4) + 0.15
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    ax.plot(
        xs_wide,
        np.power(2.0, xs_wide),
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(x0, x1)
    ax.set_ylim(0.0, y1)
    ax.grid(alpha=0.2)
    _strip_work_effort(ax)
    frames26.append(_exp_snap(fig, ax))

save_gif(frames26, "49_exponential_transition.gif", duration=TRANS_MS)

# --- 29–30: static (-4.5, 6) reference plots with legends ---
xs_leg = np.linspace(-4.5, 6.0, 700)
fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
ax.plot(
    xs_leg,
    np.power(2.0, xs_leg),
    color=CURVE_COLOR,
    linestyle="-",
    linewidth=2.0,
    label=r"$f(x)=2^x$",
)
ax.set_xlim(-4.5, 6.0)
ax.set_ylim(0.0, 2**(6.0 - 1.4) + 0.15)
ax.grid(alpha=0.2)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
_strip_work_effort(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "50_exponential_2x_legend.png")

fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
ax.plot(
    xs_leg,
    np.exp(xs_leg),
    color=CURVE_COLOR,
    linestyle="-",
    linewidth=2.0,
    label=r"$f(x)=e^x$",
)
ax.set_xlim(-4.5, 6.0)
ax.set_ylim(0.0, float(np.exp(6.0 - 1.4)) + 0.15)
ax.grid(alpha=0.2)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax.legend(loc="upper left", prop={"size": LEGEND_SIZE})
_strip_work_effort(ax)
finalize_exponential_figure(fig)
_save_exp_png(fig, "51_exponential_ex_legend.png")

# --- 66: morph (L x, 2^x) vs fixed ((ln2)x, 2^x); same box as 42/43 + conversion in legend ---
from matplotlib.lines import Line2D

_ln2 = float(np.log(2.0))


def _ln66_smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _draw_ln2_axis_morph(ax, s):
    """s in [0,1]: blue L: 1->ln2; gray is ((ln2)x, 2^x). Same box as 42/43; legend shows conversion only."""
    u = _ln66_smoothstep(s)
    L = 1.0 + u * (_ln2 - 1.0)
    y = np.power(2.0, xs_leg)
    xh_m = xs_leg * L
    xh_t = xs_leg * _ln2
    ax.clear()
    ax.plot(
        xh_t,
        y,
        color="#8c8c8c",
        linestyle=(0, (5, 3)),
        linewidth=1.65,
        zorder=1,
    )
    ax.plot(
        xh_m,
        y,
        color=CURVE_COLOR,
        linestyle="-",
        linewidth=2.0,
        zorder=2,
    )
    ax.set_xlim(-4.5, 6.0)
    ax.set_ylim(0.0, 2 ** (6.0 - 1.4) + 0.15)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    h_form = Line2D(
        [],
        [],
        linestyle="none",
        linewidth=0,
        color="none",
        label=r"$2^x=e^{(\ln 2)\,x}$",
    )
    ax.legend(handles=[h_form], loc="upper left", prop={"size": LEGEND_SIZE})
    _strip_work_effort(ax)
LN66_MS = 52
LN66_HOLD = 10
LN66_MORPH = 52
LN66_HOLD_END = 18
frames_ln66 = []
for _ in range(LN66_HOLD):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, 0.0)
    frames_ln66.append(_exp_snap(fig, ax))
for i in range(LN66_MORPH):
    t = i / (LN66_MORPH - 1) if LN66_MORPH > 1 else 1.0
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, t)
    frames_ln66.append(_exp_snap(fig, ax))
for _ in range(LN66_HOLD_END):
    fig, ax = plt.subplots(figsize=EXP_FIGSIZE)
    _draw_ln2_axis_morph(ax, 1.0)
    frames_ln66.append(_exp_snap(fig, ax))
save_gif(frames_ln66, "52_two_pow_axis_ln2_to_exp.gif", duration=LN66_MS)


# --- 44–45: softmax on exp(ST−EL); same duo layout / timing as 32–33 (Scene 6b helpers).
_EXP_YLABEL = r"$e^{\mathrm{ST}-\mathrm{EL}}$"

# Y-axis wording for PF / 48 softmax right panel (not 45–47)
_EXP_YLABEL_INLINE = r"$\mathrm{effort} = e^{\mathrm{ST}-\mathrm{EL}}$"


def _ylim_exp_twin(z_a, z_b):
    m = max(float(np.exp(z_a)), float(np.exp(z_b)))
    return 0.0, max(m * 1.12, 0.5)


def _ylim_exp_stack(z_a, z_b):
    t = float(np.exp(z_a) + np.exp(z_b))
    return 0.0, max(t * 1.08, 0.5)


def _style_twin_exp_axis(ax, ylo, yhi, ylabel, hide_xticks=False):
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel(ylabel, fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_xlabel("")
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    ax.grid(axis="y", alpha=0.22)
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.set_xlim(*N_BAR_XLIM)
    if hide_xticks:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.tick_params(axis="x", bottom=False, labelbottom=False)
    else:
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["A", "B"])


STACK_BAR_W = float(N_STACK_WIDTH * 1.14)
TWIN_EXP_BAR_W = 0.56


def _mtex(s):
    """Single mathtext string (wrap in $...$ if needed)."""
    s = str(s).strip()
    if s.startswith("$") and s.endswith("$"):
        return s
    return f"${s}$"


def _bar_mtext(ax, x, y, tex, fontsize, z=NORM_STACK_LABEL_Z, color="#141414", path_effects=None):
    kw = dict(ha="center", va="center", fontsize=fontsize, color=color, zorder=z)
    if path_effects is not None:
        kw["path_effects"] = path_effects
    ax.text(x, y, _mtex(tex), **kw)


def _stack_ab_arrows(
    ax,
    xb,
    h1,
    h2,
    labels=(r"$A$", r"$B$"),
    bar_w=None,
    dx_a=0.62,
    dx_b=None,
    reflect_a=False,
    arrows_b_only=False,
):
    """Angled arrow: tip on bar left edge at segment mid-height; label outside left."""
    bw = float(STACK_BAR_W if bar_w is None else bar_w)
    x_left = float(xb) - 0.5 * bw
    y_lo = 0.5 * h1
    y_hi = h1 + 0.5 * h2
    dxb = float(dx_a if dx_b is None else dx_b)
    dxa = float(dx_a)
    dy_a_up = 0.14 * max(h1, 0.35)
    dy_a_down = 0.21 * max(h1, 0.35)
    y_a_text = y_lo - dy_a_down if reflect_a else y_lo + dy_a_up
    if not arrows_b_only:
        ax.annotate(
            labels[0],
            xy=(x_left, y_lo),
            xytext=(x_left - dxa, y_a_text),
            textcoords="data",
            arrowprops=dict(
                arrowstyle="-|>",
                color="#333333",
                lw=0.95,
                shrinkA=2,
                shrinkB=1,
                mutation_scale=9,
            ),
            fontsize=NOTE_SIZE + 4,
            ha="right",
            va="center",
            color="#141414",
            zorder=NORM_STACK_LABEL_Z + 1,
            annotation_clip=False,
        )
    ax.annotate(
        labels[1],
        xy=(x_left, y_hi),
        xytext=(x_left - dxb, y_hi - 0.14 * max(h2, 0.35)),
        textcoords="data",
        arrowprops=dict(
            arrowstyle="-|>",
            color="#333333",
            lw=0.95,
            shrinkA=2,
            shrinkB=1,
            mutation_scale=9,
        ),
        fontsize=NOTE_SIZE + 4,
        ha="right",
        va="center",
        color="#141414",
        zorder=NORM_STACK_LABEL_Z + 1,
        annotation_clip=False,
    )


def _draw_twin_exp_bars(
    ax,
    z_a,
    z_b,
    c_a,
    c_b,
    phase,
    alpha_a=1.0,
    ylabel=_EXP_YLABEL,
    inline_exp_overlay=None,
    hide_xticks=False,
    b_overlay_only=False,
    bar_mtext_white=False,
):
    ax.clear()
    ea, eb = float(np.exp(z_a)), float(np.exp(z_b))
    ylo, yhi = _ylim_exp_twin(z_a, z_b)
    bw = TWIN_EXP_BAR_W if inline_exp_overlay else 0.48
    if phase >= 1:
        ax.bar([0], [ea], width=bw, color=[c_a], edgecolor="black", linewidth=0.9, alpha=alpha_a)
    if phase >= 2:
        ax.bar([1], [eb], width=bw, color=[c_b], edgecolor="black", linewidth=0.9)
    _style_twin_exp_axis(ax, ylo, yhi, ylabel, hide_xticks=hide_xticks)
    _norm_work_effort_gloss(ax, "effort")
    _bc = "#ffffff" if bar_mtext_white else "#141414"
    _bpe = NORM_STACK_WHITE_PE if bar_mtext_white else None
    if inline_exp_overlay == "digit_a" and phase >= 1 and not b_overlay_only:
        zi = int(round(float(z_a)))
        _bar_mtext(ax, 0.0, 0.5 * ea, str(zi), NOTE_SIZE + 12)
    elif inline_exp_overlay == "exp_pair" and phase >= 2:
        ia, ib = int(round(float(z_a))), int(round(float(z_b)))
        if b_overlay_only:
            _bar_mtext(
                ax,
                1.0,
                0.5 * eb,
                rf"e^{{{ib}}}",
                NOTE_SIZE + 11,
                color=_bc,
                path_effects=_bpe,
            )
        else:
            _bar_mtext(ax, 0.0, 0.5 * ea, rf"e^{{{ia}}}", NOTE_SIZE + 11)
            _bar_mtext(ax, 1.0, 0.5 * eb, rf"e^{{{ib}}}", NOTE_SIZE + 11)


def _parse_pf_eq_line(line):
    """Return (arrow_mathtext, bar_mathtext) from a full equation line."""
    s = line.strip()
    if not (s.startswith("$") and s.endswith("$")):
        return r"$\,$", s
    inner = s[1:-1]
    if " = " not in inner:
        return r"$\,$", s
    a, b = inner.split(" = ", 1)
    arrow = f"${a.strip()}$"
    rhs = b.strip()
    if rhs.startswith("$") and rhs.endswith("$"):
        bar = rhs
    else:
        bar = f"${rhs}$"
    return arrow, bar


def _draw_pf_stack_caption(ax_r, caption_body, h1, h2):
    lines = [ln for ln in caption_body.splitlines() if ln.strip()]
    if len(lines) < 2:
        return
    al0, br0 = _parse_pf_eq_line(lines[0])
    al1, br1 = _parse_pf_eq_line(lines[1])
    _bar_mtext(ax_r, N_STACK_X, 0.5 * h1, br0, NOTE_SIZE + 9)
    _bar_mtext(ax_r, N_STACK_X, h1 + 0.5 * h2, br1, NOTE_SIZE + 9)
    xb = float(N_STACK_X)
    bw = float(STACK_BAR_W)
    x_left = xb - 0.5 * bw
    y_lo = 0.5 * h1
    y_hi = h1 + 0.5 * h2
    dx = 0.72
    ax_r.annotate(
        al0,
        xy=(x_left, y_lo),
        xytext=(x_left - dx, y_lo - 0.21 * max(h1, 0.35)),
        textcoords="data",
        arrowprops=dict(
            arrowstyle="-|>",
            color="#333333",
            lw=0.9,
            shrinkA=2,
            shrinkB=1,
            mutation_scale=8,
        ),
        fontsize=NOTE_SIZE + 1,
        ha="right",
        va="center",
        color="#141414",
        zorder=NORM_STACK_LABEL_Z + 1,
        annotation_clip=False,
    )
    ax_r.annotate(
        al1,
        xy=(x_left, y_hi),
        xytext=(x_left - dx, y_hi - 0.14 * max(h2, 0.35)),
        textcoords="data",
        arrowprops=dict(
            arrowstyle="-|>",
            color="#333333",
            lw=0.9,
            shrinkA=2,
            shrinkB=1,
            mutation_scale=8,
        ),
        fontsize=NOTE_SIZE + 1,
        ha="right",
        va="center",
        color="#141414",
        zorder=NORM_STACK_LABEL_Z + 1,
        annotation_clip=False,
    )


def _e_frac_tex(z):
    zi = int(round(float(z)))
    return rf"e^{{{zi}}}"


def _draw_exp_softmax_stack(
    ax_r,
    stage,
    z_a,
    z_b,
    c_a,
    c_b,
    inline_bar_caption=False,
    exp_ylabel=None,
    stack_arrow_reflect_a=False,
    stack_arrow_dx_b=None,
    inline_softmax_b_only=False,
    bar_mtext_white=False,
):
    h1 = float(np.exp(z_a))
    h2 = float(np.exp(z_b))
    ylo, yhi = _ylim_exp_stack(z_a, z_b)
    ax_r.clear()
    w = STACK_BAR_W if inline_bar_caption else N_STACK_WIDTH
    ax_r.bar([N_STACK_X], [h1], width=w, bottom=0.0, color=c_a, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=w, bottom=h1, color=c_b, edgecolor="black", linewidth=0.9)
    ylab = exp_ylabel if exp_ylabel is not None else _EXP_YLABEL
    b_only = bool(inline_bar_caption and inline_softmax_b_only)
    _hi_kw = (
        dict(color="#ffffff", path_effects=NORM_STACK_WHITE_PE)
        if bar_mtext_white and b_only
        else {}
    )
    if inline_bar_caption:
        _style_twin_exp_axis(ax_r, ylo, yhi, ylab, hide_xticks=True)
        if stage <= 0:
            _stack_ab_arrows(
                ax_r,
                float(N_STACK_X),
                h1,
                h2,
                bar_w=w,
                reflect_a=stack_arrow_reflect_a,
                dx_b=stack_arrow_dx_b,
                arrows_b_only=b_only,
            )
            return
        ta, tb = _e_frac_tex(z_a), _e_frac_tex(z_b)
        den = f"{ta}+{tb}"
        p_lo = float(h1 / (h1 + h2))
        p_hi = float(h2 / (h1 + h2))
        if stage == 1:
            f_lo = r"$\dfrac{" + ta + "}{" + den + "}$"
            f_hi = r"$\dfrac{" + tb + "}{" + den + "}$"
            if not b_only:
                _bar_mtext(ax_r, N_STACK_X, 0.5 * h1, f_lo, NOTE_SIZE + 9)
            _bar_mtext(ax_r, N_STACK_X, h1 + 0.5 * h2, f_hi, NOTE_SIZE + 9, **_hi_kw)
        elif stage == 2:
            if not b_only:
                _bar_mtext(ax_r, N_STACK_X, 0.5 * h1, rf"$\approx {p_lo:.3f}$", NOTE_SIZE + 10)
            _bar_mtext(
                ax_r,
                N_STACK_X,
                h1 + 0.5 * h2,
                rf"$\approx {p_hi:.3f}$",
                NOTE_SIZE + 10,
                **_hi_kw,
            )
        else:
            if not b_only:
                _bar_mtext(ax_r, N_STACK_X, 0.5 * h1, rf"${p_lo:.3f}$", NOTE_SIZE + 10)
            _bar_mtext(ax_r, N_STACK_X, h1 + 0.5 * h2, rf"${p_hi:.3f}$", NOTE_SIZE + 10, **_hi_kw)
        _stack_ab_arrows(
            ax_r,
            float(N_STACK_X),
            h1,
            h2,
            bar_w=w,
            reflect_a=stack_arrow_reflect_a,
            dx_b=stack_arrow_dx_b,
            arrows_b_only=b_only,
        )
        return
    _style_twin_exp_axis(ax_r, ylo, yhi, ylabel=ylab)
    if stage <= 0:
        _norm_work_effort_gloss(ax_r, "effort_stack")
        return
    _norm_work_effort_gloss(ax_r, "effort_normalize")
    ta, tb = _e_frac_tex(z_a), _e_frac_tex(z_b)
    den = f"{ta}+{tb}"
    p_lo = float(h1 / (h1 + h2))
    p_hi = float(h2 / (h1 + h2))
    if stage == 1:
        body = (
            r"$\mathrm{A}:\ \dfrac{" + ta + "}{" + den + r"}$"
            + "\n"
            + r"$\mathrm{B}:\ \dfrac{" + tb + "}{" + den + r"}$"
        )
    elif stage == 2:
        body = (
            rf"$\mathrm{{A}}:\ \approx {p_lo:.3f}$"
            + "\n"
            + rf"$\mathrm{{B}}:\ \approx {p_hi:.3f}$"
        )
    else:
        body = (
            rf"$\mathrm{{A}}:\ {p_lo:.3f}$"
            + "\n"
            + rf"$\mathrm{{B}}:\ {p_hi:.3f}$"
        )
    ax_r.text(
        0.98,
        0.98,
        body,
        transform=ax_r.transAxes,
        ha="right",
        va="top",
        fontsize=NOTE_SIZE + 6,
        linespacing=1.42,
        color="#141414",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="white",
            edgecolor="#c4c4c4",
            linewidth=0.75,
            alpha=0.96,
        ),
        zorder=NORM_STACK_LABEL_Z,
    )


def _softmax_cadence_core(
    ax0_l,
    ax0_r,
    fig0,
    left_draw,
    z_a,
    z_b,
    c_st_a,
    c_st_b,
    c_exp_a,
    c_exp_b,
    twin_alpha_a,
    twin_z_ylim=None,
    right_inline_softmax=False,
    stack_exp_ylabel=None,
    stack_arrow_reflect_a=False,
    stack_arrow_dx_b=None,
    inline_softmax_b_only=False,
    skip_twin_work_lead=False,
    stel_digit_white=False,
    stel_omit_zero_b=False,
):
    b_overlay = bool(right_inline_softmax and inline_softmax_b_only)
    bar_tex_white = bool(right_inline_softmax and inline_softmax_b_only)
    out = []
    if not skip_twin_work_lead:
        for _ in range(4):
            left_draw(ax0_l, 0)
            _draw_twin_bars(
                ax0_r,
                z_a,
                z_b,
                c_st_a,
                c_st_b,
                phase=0,
                z_ylim=twin_z_ylim,
                work_effort_gloss=True,
                inline_stel_z_digits=right_inline_softmax,
                stel_digits_b_only=b_overlay,
                stel_digit_white=stel_digit_white,
                stel_omit_zero_b=stel_omit_zero_b,
            )
            out.append(_norm_frame_png(fig0))
        for _ in range(5):
            left_draw(ax0_l, 1)
            _draw_twin_bars(
                ax0_r,
                z_a,
                z_b,
                c_st_a,
                c_st_b,
                phase=1,
                alpha_a=twin_alpha_a,
                z_ylim=twin_z_ylim,
                work_effort_gloss=True,
                inline_stel_z_digits=right_inline_softmax,
                stel_digits_b_only=b_overlay,
                stel_digit_white=stel_digit_white,
                stel_omit_zero_b=stel_omit_zero_b,
            )
            out.append(_norm_frame_png(fig0))
        for _ in range(5):
            left_draw(ax0_l, 2)
            _draw_twin_bars(
                ax0_r,
                z_a,
                z_b,
                c_st_a,
                c_st_b,
                phase=2,
                alpha_a=twin_alpha_a,
                z_ylim=twin_z_ylim,
                work_effort_gloss=True,
                inline_stel_z_digits=right_inline_softmax,
                stel_digits_b_only=b_overlay,
                stel_digit_white=stel_digit_white,
                stel_omit_zero_b=stel_omit_zero_b,
            )
            out.append(_norm_frame_png(fig0))
    if not right_inline_softmax:
        for _ in range(5):
            left_draw(ax0_l, 2)
            _draw_twin_exp_bars(
                ax0_r,
                z_a,
                z_b,
                c_exp_a,
                c_exp_b,
                phase=1,
                alpha_a=twin_alpha_a,
                ylabel=_EXP_YLABEL,
                inline_exp_overlay=None,
            )
            out.append(_norm_frame_png(fig0))
    for _ in range(5):
        left_draw(ax0_l, 2)
        _draw_twin_exp_bars(
            ax0_r,
            z_a,
            z_b,
            c_exp_a,
            c_exp_b,
            phase=2,
            alpha_a=twin_alpha_a,
            ylabel=(stack_exp_ylabel if stack_exp_ylabel is not None else _EXP_YLABEL_INLINE)
            if right_inline_softmax
            else _EXP_YLABEL,
            inline_exp_overlay="exp_pair" if right_inline_softmax else None,
            b_overlay_only=b_overlay,
            bar_mtext_white=bar_tex_white,
        )
        out.append(_norm_frame_png(fig0))
    for st in (0, 1, 2, 3):
        nrep = 6 if st == 3 else 5
        for _ in range(nrep):
            left_draw(ax0_l, 2)
            _draw_exp_softmax_stack(
                ax0_r,
                st,
                z_a,
                z_b,
                c_exp_a,
                c_exp_b,
                inline_bar_caption=right_inline_softmax,
                exp_ylabel=stack_exp_ylabel
                if right_inline_softmax
                else None,
                stack_arrow_reflect_a=stack_arrow_reflect_a
                if right_inline_softmax
                else False,
                stack_arrow_dx_b=stack_arrow_dx_b if right_inline_softmax else None,
                inline_softmax_b_only=inline_softmax_b_only,
                bar_mtext_white=bar_tex_white,
            )
            out.append(_norm_frame_png(fig0))
    return out


def _frames_exp_softmax_same_cadence(
    ax0_l,
    ax0_r,
    fig0,
    s_a,
    e_a,
    y_a,
    c_a_plane,
    s_b,
    e_b,
    y_b,
    c_b_plane,
    z_a,
    z_b,
    c_st_a,
    c_st_b,
    c_exp_a,
    c_exp_b,
    twin_alpha_a,
    right_inline_softmax=False,
    stack_exp_ylabel=None,
    stack_arrow_reflect_a=False,
    stack_arrow_dx_b=None,
    inline_softmax_b_only=False,
    skip_twin_work_lead=False,
    stel_digit_white=False,
    stel_omit_zero_b=False,
):
    def _ld(ax, ph):
        _draw_left_progress(ax, s_a, e_a, y_a, c_a_plane, s_b, e_b, y_b, c_b_plane, phase=ph)

    return _softmax_cadence_core(
        ax0_l,
        ax0_r,
        fig0,
        _ld,
        z_a,
        z_b,
        c_st_a,
        c_st_b,
        c_exp_a,
        c_exp_b,
        twin_alpha_a,
        None,
        right_inline_softmax,
        stack_exp_ylabel,
        stack_arrow_reflect_a,
        stack_arrow_dx_b,
        inline_softmax_b_only,
        skip_twin_work_lead,
        stel_digit_white,
        stel_omit_zero_b,
    )


frames_exp_pos = []
fig_ep, (ax_ep_l, ax_ep_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_ep.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
_ce_pos_a, _ce_pos_b = _exp_bar_facecolors([ZA1, ZB4])
frames_exp_pos.extend(
    _frames_exp_softmax_same_cadence(
        ax_ep_l,
        ax_ep_r,
        fig_ep,
        SA1,
        EA1,
        YA1,
        COL_POS_LO,
        SB4,
        EB4,
        YB4,
        COL_POS_HI,
        ZA1,
        ZB4,
        COL_POS_LO,
        COL_POS_HI,
        _ce_pos_a,
        _ce_pos_b,
        1.0,
        right_inline_softmax=True,
        stack_exp_ylabel=_EXP_YLABEL_INLINE,
        stack_arrow_reflect_a=True,
        stack_arrow_dx_b=0.72,
        inline_softmax_b_only=True,
        stel_digit_white=True,
    )
)
plt.close(fig_ep)
save_gif(frames_exp_pos, "53_norm_exp_positive_softmax.gif", duration=NORM_MS)

frames_exp_neg = []
fig_en, (ax_en_l, ax_en_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_en.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
c_na, c_nb = _exp_bar_facecolors([ZAN, ZB4])
frames_exp_neg.extend(
    _frames_exp_softmax_same_cadence(
        ax_en_l,
        ax_en_r,
        fig_en,
        SAN,
        EAN,
        YAN,
        COL_NEG_LO,
        SB4,
        EB4,
        YB4,
        COL_POS_HI,
        ZAN,
        ZB4,
        COL_NEG_LO,
        COL_POS_HI,
        c_na,
        c_nb,
        0.62,
        right_inline_softmax=True,
        stack_exp_ylabel=_EXP_YLABEL_INLINE,
        stack_arrow_reflect_a=True,
        stack_arrow_dx_b=0.72,
        inline_softmax_b_only=True,
        stel_digit_white=True,
    )
)
plt.close(fig_en)
save_gif(frames_exp_neg, "54_norm_exp_negative_softmax.gif", duration=NORM_MS)

# --- 46–47: far-region zoom-out; duo A/B with ST−EL gaps 101 vs 104; softmax when B on ST−EL=0 and z_A=1 ---
ZOOM_MS = 45
N_ZOOM = 52
ZOOM_X_END = (4.0, 120.0)
ZOOM_Y_END = (4.0, 110.0)
Z_GAP_A, Z_GAP_B = 101.0, 104.0
S_FAR_A, S_FAR_B = 110.0, 115.0
E_FAR_A = S_FAR_A - float(midpoint_shift) - Z_GAP_A
E_FAR_B = S_FAR_B - float(midpoint_shift) - Z_GAP_B
TWIN_Z_LIM_FAR = (0.0, max(Z_GAP_A, Z_GAP_B) * 1.12)


def _threshold_line_xy(ax, xlo, xhi, shift, label="ST - EL = 0"):
    x = np.linspace(xlo, xhi, 600)
    ax.plot(x, x - shift, "--", color="black", linewidth=1.0, label=label)


def _lerp(a0, a1, t):
    return float(a0 + t * (a1 - a0))


def _smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _frame_zoom_far(ax, x0, x1, y0, y1):
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real)
    add_outcome_icon(ax, S_FAR_A, E_FAR_A, passed=True, alpha=0.98, zoom=0.2)
    add_outcome_icon(ax, S_FAR_B, E_FAR_B, passed=True, alpha=0.98, zoom=0.2)
    _threshold_line_xy(ax, x0, x1, float(midpoint_shift), label="ST - EL = 0")
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")


def _draw_far_pair_left(ax, phase):
    sh = float(midpoint_shift)
    ax.clear()
    draw_dataset(ax, study_real, exam_real, y_real, alpha=0.22)
    add_outcome_icon(ax, S_FAR_A, E_FAR_A, passed=True, alpha=0.98, zoom=0.2)
    add_outcome_icon(ax, S_FAR_B, E_FAR_B, passed=True, alpha=0.98, zoom=0.2)
    _threshold_line_xy(ax, ZOOM_X_END[0], ZOOM_X_END[1], sh, label="ST - EL = 0")
    if phase >= 1:
        _draw_student_distance(ax, S_FAR_A, E_FAR_A, 1, COL_POS_LO)
    if phase >= 2:
        _draw_student_distance(ax, S_FAR_B, E_FAR_B, 1, COL_POS_HI)
    ax.set_xlim(*ZOOM_X_END)
    ax.set_ylim(*ZOOM_Y_END)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")


def _blank_twin_st_axis(ax_r):
    ax_r.clear()
    _style_bar_axis(ax_r, stacked=False, ylabel="ST - EL", z_ylim=TWIN_Z_LIM_FAR)
    _norm_work_effort_gloss(ax_r, "work")


frames_far = []
Z_FAR_A = float(S_FAR_A - E_FAR_A - float(midpoint_shift))
Z_FAR_B = float(S_FAR_B - E_FAR_B - float(midpoint_shift))
assert np.isclose(Z_FAR_A, Z_GAP_A) and np.isclose(Z_FAR_B, Z_GAP_B)
_ce_fa, _ce_fb = _exp_bar_facecolors([Z_FAR_A, Z_FAR_B])
fig_fz, (ax_fz_l, ax_fz_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_fz.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
for i in range(N_ZOOM):
    u = _smoothstep(i / (N_ZOOM - 1)) if N_ZOOM > 1 else 1.0
    xa = _lerp(xlim[0], ZOOM_X_END[0], u)
    xb = _lerp(xlim[1], ZOOM_X_END[1], u)
    ya = _lerp(ylim[0], ZOOM_Y_END[0], u)
    yb = _lerp(ylim[1], ZOOM_Y_END[1], u)
    _frame_zoom_far(ax_fz_l, xa, xb, ya, yb)
    _blank_twin_st_axis(ax_fz_r)
    frames_far.append(_norm_frame_png(fig_fz))
for _ in range(5):
    _draw_far_pair_left(ax_fz_l, 1)
    _draw_twin_bars(
        ax_fz_r,
        Z_FAR_A,
        Z_FAR_B,
        COL_POS_LO,
        COL_POS_HI,
        phase=1,
        z_ylim=TWIN_Z_LIM_FAR,
        work_effort_gloss=True,
        inline_stel_z_digits=True,
        stel_digit_white=True,
    )
    frames_far.append(_norm_frame_png(fig_fz))
for _ in range(5):
    _draw_far_pair_left(ax_fz_l, 2)
    _draw_twin_bars(
        ax_fz_r,
        Z_FAR_A,
        Z_FAR_B,
        COL_POS_LO,
        COL_POS_HI,
        phase=2,
        z_ylim=TWIN_Z_LIM_FAR,
        work_effort_gloss=True,
        inline_stel_z_digits=True,
        stel_digit_white=True,
    )
    frames_far.append(_norm_frame_png(fig_fz))


def _ld_far_locked(ax, ph):
    _draw_far_pair_left(ax, ph)


frames_far.extend(
    _softmax_cadence_core(
        ax_fz_l,
        ax_fz_r,
        fig_fz,
        _ld_far_locked,
        Z_FAR_A,
        Z_FAR_B,
        COL_POS_LO,
        COL_POS_HI,
        _ce_fa,
        _ce_fb,
        1.0,
        TWIN_Z_LIM_FAR,
        right_inline_softmax=True,
        stack_exp_ylabel=_EXP_YLABEL_INLINE,
        stack_arrow_reflect_a=True,
        stack_arrow_dx_b=0.72,
        inline_softmax_b_only=True,
        skip_twin_work_lead=True,
        stel_digit_white=True,
    )
)
plt.close(fig_fz)
save_gif(frames_far, "55_far_study_hours_zoom.gif", duration=NORM_MS)

# Same (0,7) scatter as 44/45: A reuses the z=+1 pass pick; B is synthetic on ST−EL=0 (not in the point list).
S_Z1, E_Z1, Y_Z1 = SA1, EA1, YA1
sh_z = float(midpoint_shift)
S_TH, E_TH, Y_TH = 5.0, 5.0 - sh_z, 1
Z_TH_A = float(S_Z1 - E_Z1 - sh_z)
Z_TH_B = float(S_TH - E_TH - sh_z)
assert np.isclose(Z_TH_A, ZA1) and np.isclose(Z_TH_B, 0.0)
_ce_th_a, _ce_th_b = _exp_bar_facecolors([Z_TH_A, Z_TH_B])
frames_th = []
fig_th, (ax_th_l, ax_th_r) = plt.subplots(
    1,
    2,
    figsize=NORM_DUO_FIGSIZE,
    gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
)
fig_th.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
frames_th.extend(
    _frames_exp_softmax_same_cadence(
        ax_th_l,
        ax_th_r,
        fig_th,
        S_Z1,
        E_Z1,
        Y_Z1,
        COL_POS_LO,
        S_TH,
        E_TH,
        Y_TH,
        COL_POS_HI,
        Z_TH_A,
        Z_TH_B,
        COL_POS_LO,
        COL_POS_HI,
        _ce_th_a,
        _ce_th_b,
        1.0,
        right_inline_softmax=True,
        stack_exp_ylabel=_EXP_YLABEL_INLINE,
        stack_arrow_reflect_a=True,
        stack_arrow_dx_b=0.72,
        stel_omit_zero_b=True,
    )
)
plt.close(fig_th)
save_gif(frames_th, "56_softmax_threshold_and_z1.gif", duration=NORM_MS)



def _draw_pf_softmax_stack_snapshot(ax_l, ax_r, caption_body):
    """Same duo as 47 @ phase 2 (left); stacked exp bars; probabilities as in-bar mathtext (49–53)."""
    _draw_left_progress(
        ax_l,
        S_Z1,
        E_Z1,
        Y_Z1,
        COL_POS_LO,
        S_TH,
        E_TH,
        Y_TH,
        COL_POS_HI,
        phase=2,
    )
    z_a, z_b = float(Z_TH_A), float(Z_TH_B)
    h1 = float(np.exp(z_a))
    h2 = float(np.exp(z_b))
    ylo, yhi = _ylim_exp_stack(z_a, z_b)
    ax_r.clear()
    ax_r.bar([N_STACK_X], [h1], width=STACK_BAR_W, bottom=0.0, color=_ce_th_a, edgecolor="black", linewidth=0.9)
    ax_r.bar([N_STACK_X], [h2], width=STACK_BAR_W, bottom=h1, color=_ce_th_b, edgecolor="black", linewidth=0.9)
    _style_twin_exp_axis(ax_r, ylo, yhi, _EXP_YLABEL_INLINE, hide_xticks=True)
    _draw_pf_stack_caption(ax_r, caption_body, h1, h2)

_pf_cap_pass = r"$\mathrm{P}(\mathrm{A\ passes}) = \dfrac{e^{1}}{e^{0}+e^{1}}$"
_pf_snap_specs = (
    (
        "57_softmax_pf_stack_normalized_frac.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{0}}{e^{0}+e^{1}}$",
    ),
    (
        "58_softmax_pf_stack_fail_sum_ratio.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{0}+e^{1}-e^{1}}{e^{0}+e^{1}}$",
    ),
    (
        "59_softmax_pf_stack_fail_one_minus.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = 1 - \mathrm{P}(\mathrm{A\ passes})$",
    ),
    (
        "60_softmax_pf_stack_fail_eneg1.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{A\ fails}) = \dfrac{e^{-1}}{e^{-1}+e^{0}}$",
    ),
    (
        "61_softmax_pf_stack_fail_sum_ratio_pass_B.png",
        _pf_cap_pass + "\n" + r"$\mathrm{P}(\mathrm{B}) = \dfrac{e^{0}}{e^{0}+e^{1}}$",
    ),
)

for _pf_fn, _pf_body in _pf_snap_specs:
    _fig_pf, (_ax_pf_l, _ax_pf_r) = plt.subplots(
        1,
        2,
        figsize=NORM_DUO_FIGSIZE,
        gridspec_kw={"width_ratios": [1.12, 1.0], "wspace": 0.28},
    )
    _fig_pf.subplots_adjust(left=0.07, right=0.98, top=0.96, bottom=0.12)
    _draw_pf_softmax_stack_snapshot(_ax_pf_l, _ax_pf_r, _pf_body)
    _im_pf = _norm_frame_png(_fig_pf)
    _im_pf.save(OUTPUT_DIR / _pf_fn)
    plt.close(_fig_pf)




**How to run Scene 8:** run **Cell 1** (paths, `sigmoid`, `save_gif`, `fig_to_image`, typography) and **Cells 2–3** (datasets and plane helpers) first. Then run **Scene 8 — Setup (shared)** once. Each following `###` section is a **single export** you can run on its own.

## Scene 8 - Sigmoid in 3D (pass/fail mirrored surfaces)

Script alignment:
- map any `ST - EL` value to pass probability with sigmoid
- show mirrored fail probability
- present both in 3D style
- **`77_from27_topdown_reveal_lift_3d_reverse64.gif`**: **Half-plane** on **amplified** σ (**g = 5**), **2D shrink**, **3D** top, **−45°** azimuth at top, **tilt**, **360°** spin (**+45°** vs a plain **315°** orbit so final bearing matches), **de-amplify** (**g → ~0**, field → **~½**), end; **3D** frames omit axis titles


### Scene 8 — Setup (shared)

Meshes `ST`/`EL`, colormap `CMAP`, 2D sigmoid helpers, 3D styling (`style_sigmoid_axes`, `scatter_whole_data_by_class`, `sig3d_frame_png`), orbit angles, top-down camera schedule, and pass/fail point heights. **Run this cell once** before any export cell below.


In [47]:
import io
import matplotlib as mpl
from PIL import Image
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
st_axis = np.linspace(xlim[0], xlim[1], 220)
el_axis = np.linspace(ylim[0], ylim[1], 220)
ST, EL = np.meshgrid(st_axis, el_axis)
DIFF = ST - EL
P_PASS = sigmoid(DIFF)
P_FAIL = sigmoid(-DIFF)
cvals  = [0, .5, 1]
colors = ['red', 'white', 'green']
norm=plt.Normalize(min(cvals),max(cvals))
tuples = list(zip(map(norm,cvals), colors))
CMAP = mpl.colors.LinearSegmentedColormap.from_list("", tuples, 100)

# Threshold curve where ST=EL and sigmoid(0)=1/2
diag_line = np.linspace(max(xlim[0], ylim[0]), min(xlim[1], ylim[1]), 220)
threshold_half = np.full_like(diag_line, 0.5)

# Reference cross-section for readability (fixed exam length)
el_ref = np.full_like(st_axis, np.mean(ylim))

all_study = study_real
all_exam = exam_real
all_diff = all_study - all_exam

# Match the red/green palette used in neural-networks.ipynb (21.gif section)
SIG_PASS_COLOR = "green"
SIG_FAIL_COLOR = "red"


def style_sigmoid_axes(
    ax,
    az,
    elev=26,
    hide_z=False,
    exam_label_2d=False,
    diag_prob_scale=1.0,
):
    _dps = float(diag_prob_scale)
    ax.plot(
        diag_line,
        diag_line,
        _dps * threshold_half,
        color="black",
        linestyle="--",
        linewidth=1,
    )
    ax.tick_params(axis="x", labelsize=FONT_SIZE)
    ax.tick_params(axis="y", labelsize=FONT_SIZE)
    if hide_z:
        ax.set_zticks([])
        ax.set_zticklabels([])
        ax.tick_params(axis="z", labelsize=0, colors="none")
    else:
        ax.tick_params(axis="z", labelsize=FONT_SIZE)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=12.5)
    if exam_label_2d:
        ax.set_ylabel("")
        ax.text2D(
            -0.2,
            0.5,
            "Exam length (hours)",
            transform=ax.transAxes,
            fontsize=AXIS_LABEL_SIZE,
            rotation=90,
            va="center",
            ha="center",
        )
    else:
        ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=12.5)
    ax.set_zlabel("" if hide_z else "Probability", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=elev, azim=az)


def scatter_whole_data_by_class(ax, prob_values, rotate_icons_180=False):
    """Draw pass/fail outcome icons as flat RGBA quads parallel to the XY plane at each z."""
    xr = float(xlim[1] - xlim[0])
    yr = float(ylim[1] - ylim[0])
    span = min(xr, yr) * 0.045
    nx, ny = 18, 18
    prob_values = np.asarray(prob_values, dtype=float)
    polys = []
    fcols = []
    for k in range(len(all_study)):
        s = float(all_study[k])
        e = float(all_exam[k])
        z = float(prob_values[k])
        img_arr = CHECK_ICON if int(y_real[k]) == 1 else CROSS_ICON
        im = Image.fromarray(np.asarray(img_arr, dtype=np.uint8), mode="RGBA")
        im = im.resize((nx, ny), Image.LANCZOS)
        if rotate_icons_180:
            # Horizontal flip then 180° image rotation (re-sized back to nx×ny for quad grid)
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
            im = im.rotate(
                180,
                expand=True,
                resample=Image.BICUBIC,
                fillcolor=(255, 255, 255, 0),
            )
            im = im.resize((nx, ny), Image.LANCZOS)
        pix = np.asarray(im).astype(float) / 255.0
        for jj in range(ny):
            for ii in range(nx):
                rgba = pix[jj, ii]
                if rgba[3] < 0.06:
                    continue
                x0 = s - 0.5 * span + (ii / nx) * span
                x1 = s - 0.5 * span + ((ii + 1) / nx) * span
                y0 = e - 0.5 * span + (jj / ny) * span
                y1 = e - 0.5 * span + ((jj + 1) / ny) * span
                polys.append([[x0, y0, z], [x1, y0, z], [x1, y1, z], [x0, y1, z]])
                fcols.append((rgba[0], rgba[1], rgba[2], float(rgba[3]) * 0.97))
    if polys:
        coll = Poly3DCollection(
            polys,
            facecolors=fcols,
            edgecolors="none",
            linewidths=0.0,
            shade=False,
        )
        ax.add_collection3d(coll)


ROTATION_FRAMES = 120
ROTATION_MS = 32
SIG3D_FIGSIZE = EXPORT_FIGSIZE
SIG3D_DPI = EXPORT_DPI
SIG3D_ADJ = EXPORT_ADJ
SIG3D_ROT_STEP_DEG = 0.5
SIG3D_AZ0 = 25.0
SIG3D_ORBIT_AZ = SIG3D_AZ0 + np.arange(0.0, 360.0, SIG3D_ROT_STEP_DEG, dtype=float)
Z_flat = np.zeros_like(ST, dtype=float)
zeros_pts = np.zeros_like(all_diff, dtype=float)

from matplotlib.colors import to_rgb

extra_angles = SIG3D_ORBIT_AZ
_z_pass_pts = sigmoid(all_diff)
_z_fail_pts = sigmoid(-all_diff)
_last_az_pass = float(extra_angles[-1])

SIG3D_TOP_ELEV0 = 26.0
SIG3D_TOP_ELEV1 = 89.0
TOPDOWN_FRAMES = max(2, int(np.round((SIG3D_TOP_ELEV1 - SIG3D_TOP_ELEV0) / SIG3D_ROT_STEP_DEG)) + 1)
TOPDOWN_TURN_FRAMES = max(2, int(np.round(32.0 / SIG3D_ROT_STEP_DEG)) + 1)
TOPDOWN_END_HOLD_FRAMES = 12
topdown_elevations = np.linspace(SIG3D_TOP_ELEV0, SIG3D_TOP_ELEV1, TOPDOWN_FRAMES)
topdown_azimuth = 180 + 58
counterclockwise_turn = np.linspace(topdown_azimuth, topdown_azimuth + 32.0, TOPDOWN_TURN_FRAMES)
SIG3D_MORPH_FRAMES = 100
SIG3D_FLAT_HOLD = 16


def sig3d_frame_png(fig):
    fig.subplots_adjust(**SIG3D_ADJ)
    return fig_to_image(fig, tight_layout=False)


SIG2D_FIGSIZE = EXPORT_FIGSIZE
SIG2D_DPI = EXPORT_DPI
SIG2D_X = np.linspace(-7.0, 7.0, 900)
SIG2D_Y = sigmoid(SIG2D_X)
SIG2D_Y_NEG = sigmoid(-SIG2D_X)
SIG2D_COLOR = "#2ca02c"  # green (same family as pass / 3D hints)
SIG2D_MIRROR_PASS = "#2ca02c"
SIG2D_MIRROR_FAIL = "#d62728"
SIG2D_ADJ = dict(left=0.11, right=0.97, top=0.94, bottom=0.12)

from matplotlib import lines as mlines


def _finalize_sig2d(fig):
    fig.subplots_adjust(**SIG2D_ADJ)


def _save_sig2d(fig, filename):
    _finalize_sig2d(fig)
    fig.savefig(
        OUTPUT_DIR / filename,
        format="png",
        dpi=SIG2D_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


def _sig2d_axes_and_crosshairs(ax):
    ax.set_xlim(-7.0, 7.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel(r"$z = \mathrm{ST}-\mathrm{EL}$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel(r"$\sigma(z)$", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    ax.axhline(0.5, color="black", linewidth=0.9, linestyle=":", alpha=0.55)
    ax.axvline(0.0, color="black", linewidth=0.9, linestyle=":", alpha=0.55)


def _plot_sigmoid_2d_base(ax):
    _sig2d_axes_and_crosshairs(ax)
    ax.plot(SIG2D_X, SIG2D_Y, color=SIG2D_COLOR, linewidth=2.2)


SIG2D_LEGEND_FS = LEGEND_SIZE + 6
SIG2D_LEGEND_48 = r"$\sigma(z)=\frac{e^{z}}{e^{z}+e^{0}}$"


def _legend_sig2d(ax, label_math):
    h = [mlines.Line2D([], [], linestyle="none", linewidth=0, color="none", label=label_math)]
    ax.legend(
        handles=h,
        loc="upper left",
        prop={"size": SIG2D_LEGEND_FS},
        frameon=True,
        borderaxespad=0.9,
        labelspacing=0.9,
    )


def _sig2d_frame_png(fig):
    _finalize_sig2d(fig)
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=SIG2D_DPI,
        bbox_inches=None,
        pad_inches=SAVE_PAD_INCHES,
    )
    buf.seek(0)
    im = Image.open(buf).convert("RGB").copy()
    buf.close()
    return im


def _sig2d_smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def _sig2d_draw_reveal(ax, x_right):
    """Axes, crosshairs, legend (same as 48), and sigmoid curve for z <= x_right."""
    ax.clear()
    _sig2d_axes_and_crosshairs(ax)
    _legend_sig2d(ax, SIG2D_LEGEND_48)
    m = SIG2D_X <= x_right
    if np.count_nonzero(m) >= 2:
        ax.plot(
            SIG2D_X[m],
            SIG2D_Y[m],
            color=SIG2D_COLOR,
            linewidth=2.2,
            solid_capstyle="round",
        )


SIG2D_REVEAL_MS = 40
SIG2D_REVEAL_HOLD = 16
SIG2D_REVEAL_ANIM = 88
SIG2D_REVEAL_TAIL = 18
SIG2D_HIGHLIGHT_MS = 42
SIG2D_HIGHLIGHT_HOLD = 14
SIG2D_HIGHLIGHT_STAGE = 10

# Fixed layout for 60–62: same ticks/limits every frame (no merge-based rescaling).
SIG2D_HINT_Y6 = float(sigmoid(6.0))
SIG2D_HINT_YM6 = float(sigmoid(-6.0))
SIG2D_HIGHLIGHT_XTICKS = np.arange(-7, 8, dtype=float)
# Omit 0.0 and 1.0: σ(±6) ticks sit almost on the axis ends and overlap those labels.
SIG2D_HIGHLIGHT_YTICKS = np.array(
    sorted({0.2, 0.4, 0.5, 0.6, 0.8, SIG2D_HINT_Y6, SIG2D_HINT_YM6}),
    dtype=float,
)


def _sig2d_frame_full_green(ax):
    ax.clear()
    _sig2d_axes_and_crosshairs(ax)
    _legend_sig2d(ax, SIG2D_LEGEND_48)
    ax.plot(
        SIG2D_X,
        SIG2D_Y,
        color=SIG2D_COLOR,
        linewidth=2.2,
        solid_capstyle="round",
        zorder=3,
    )


def _sig2d_highlight_ytick_label(v):
    if abs(v - SIG2D_HINT_Y6) < 1e-10:
        return f"{SIG2D_HINT_Y6:.4f}"
    if abs(v - SIG2D_HINT_YM6) < 1e-10:
        return f"{SIG2D_HINT_YM6:.4f}"
    return f"{v:.1f}"


def _sig2d_highlight_freeze_ticks(ax):
    ax.set_autoscale_on(False)
    ax.set_xlim(-7.0, 7.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks(SIG2D_HIGHLIGHT_XTICKS)
    ax.set_xticklabels([str(int(t)) for t in SIG2D_HIGHLIGHT_XTICKS])
    ax.set_yticks(SIG2D_HIGHLIGHT_YTICKS)
    ax.set_yticklabels([_sig2d_highlight_ytick_label(float(t)) for t in SIG2D_HIGHLIGHT_YTICKS])


def _bold_nearest_ticklabel(ax, axis, value, tol):
    labs = ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()
    ticks = ax.get_xticks() if axis == "x" else ax.get_yticks()
    for lab, tk in zip(labs, ticks):
        w = "bold" if abs(float(tk) - float(value)) <= tol else "normal"
        lab.set_fontweight(w)
        lab.set_fontsize(FONT_SIZE)


def _sig2d_gif_three_highlights():
    x_left = float(SIG2D_X[0])
    specs = [
        (6.0, "63_sigmoid_2d_hint_plus6h_prob_near1.gif"),
        (-6.0, "64_sigmoid_2d_hint_deficit6h_prob_near0.gif"),
        (0.0, "65_sigmoid_2d_threshold_st_eq_el_prob_half.gif"),
    ]
    for z0, fn in specs:
        y0 = float(sigmoid(z0))
        frames = []
        fig, ax = plt.subplots(figsize=SIG2D_FIGSIZE)
        try:
            for _ in range(SIG2D_HIGHLIGHT_HOLD):
                _sig2d_frame_full_green(ax)
                _sig2d_highlight_freeze_ticks(ax)
                frames.append(_sig2d_frame_png(fig))
            for st in range(4):
                for _ in range(SIG2D_HIGHLIGHT_STAGE):
                    _sig2d_frame_full_green(ax)
                    _sig2d_highlight_freeze_ticks(ax)
                    if st >= 1:
                        ax.plot([z0, z0], [0.0, y0], color="black", linewidth=1.9, zorder=5)
                    if st >= 2:
                        ax.plot([x_left, z0], [y0, y0], color="black", linewidth=1.9, zorder=5)
                    _bold_nearest_ticklabel(ax, "x", z0, tol=1e-6)
                    if st >= 3:
                        _bold_nearest_ticklabel(ax, "y", y0, tol=0.05)
                    frames.append(_sig2d_frame_png(fig))
            for _ in range(SIG2D_HIGHLIGHT_HOLD):
                _sig2d_frame_full_green(ax)
                _sig2d_highlight_freeze_ticks(ax)
                ax.plot([z0, z0], [0.0, y0], color="black", linewidth=1.9, zorder=5)
                ax.plot([x_left, z0], [y0, y0], color="black", linewidth=1.9, zorder=5)
                _bold_nearest_ticklabel(ax, "x", z0, tol=1e-6)
                _bold_nearest_ticklabel(ax, "y", y0, tol=0.05)
                frames.append(_sig2d_frame_png(fig))
        finally:
            plt.close(fig)
        save_gif(frames, fn, duration=SIG2D_HIGHLIGHT_MS)

def _sig2d_draw_mirror_reveal_frame(ax, x_fail_right):
    """Green sigma(z) always; red sigma(-z) revealed for z <= x_fail_right."""
    ax.clear()
    _sig2d_axes_and_crosshairs(ax)
    ax.plot(
        SIG2D_X,
        SIG2D_Y,
        color=SIG2D_MIRROR_PASS,
        linewidth=2.35,
        solid_capstyle="round",
        zorder=3,
    )
    m = SIG2D_X <= x_fail_right
    if np.count_nonzero(m) >= 2:
        ax.plot(
            SIG2D_X[m],
            SIG2D_Y_NEG[m],
            color=SIG2D_MIRROR_FAIL,
            linewidth=2.35,
            solid_capstyle="round",
            zorder=4,
        )
    h_p = mlines.Line2D([], [], color=SIG2D_MIRROR_PASS, linewidth=2.6, label=r"$\sigma(z)$")
    h_f = mlines.Line2D([], [], color=SIG2D_MIRROR_FAIL, linewidth=2.6, label=r"$\sigma(-z)$")
    ax.legend(handles=[h_p, h_f], loc="upper right", fontsize=LEGEND_SIZE, frameon=True, borderaxespad=0.55)


SIG2D_MIRROR_REVEAL_MS = SIG2D_REVEAL_MS


### `62_sigmoid_2d_normalized_exp_reveal.gif`

2D normalized-form sigmoid; curve reveals left→right.


In [48]:
frames_s2d_reveal = []
fig_s2d_r, ax_s2d_r = plt.subplots(figsize=SIG2D_FIGSIZE)
for _ in range(SIG2D_REVEAL_HOLD):
    ax_s2d_r.clear()
    _sig2d_axes_and_crosshairs(ax_s2d_r)
    _legend_sig2d(ax_s2d_r, SIG2D_LEGEND_48)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
for i in range(SIG2D_REVEAL_ANIM):
    u = _sig2d_smoothstep(i / (SIG2D_REVEAL_ANIM - 1)) if SIG2D_REVEAL_ANIM > 1 else 1.0
    xr = -7.0 + 14.0 * u
    _sig2d_draw_reveal(ax_s2d_r, xr)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
for _ in range(SIG2D_REVEAL_TAIL):
    _sig2d_draw_reveal(ax_s2d_r, 7.0)
    frames_s2d_reveal.append(_sig2d_frame_png(fig_s2d_r))
plt.close(fig_s2d_r)
save_gif(frames_s2d_reveal, "62_sigmoid_2d_normalized_exp_reveal.gif", duration=SIG2D_REVEAL_MS)



### `60`–`62` sigmoid 2D hint GIFs

Staged ticks and guides at **z = ±6** and **z = 0**.


In [49]:
_sig2d_gif_three_highlights()


### `66_sigmoid_sigma_negz_reveal.gif`

Mirror reveal: **σ(−z)** in red vs **σ(z)** in green.


In [50]:
frames_s2d_mirror = []
fig_s2d_m, ax_s2d_m = plt.subplots(figsize=SIG2D_FIGSIZE)
for _ in range(SIG2D_REVEAL_HOLD):
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, -7.0)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
for i in range(SIG2D_REVEAL_ANIM):
    u = _sig2d_smoothstep(i / (SIG2D_REVEAL_ANIM - 1)) if SIG2D_REVEAL_ANIM > 1 else 1.0
    xr = -7.0 + 14.0 * u
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, xr)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
for _ in range(SIG2D_REVEAL_TAIL):
    _sig2d_draw_mirror_reveal_frame(ax_s2d_m, 7.0)
    frames_s2d_mirror.append(_sig2d_frame_png(fig_s2d_m))
plt.close(fig_s2d_m)
save_gif(frames_s2d_mirror, "66_sigmoid_sigma_negz_reveal.gif", duration=SIG2D_MIRROR_REVEAL_MS)


### `64`–`66` 2D legend PNGs

Three static legend variants for the 2D sigmoid story.


In [51]:
fig_s2a, ax_s2a = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2a)
_legend_sig2d(ax_s2a, SIG2D_LEGEND_48)

_save_sig2d(fig_s2a, "67_sigmoid_2d_normalized_exp_legend.png")

fig_s2b, ax_s2b = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2b)
_legend_sig2d(
    ax_s2b,
    r"$\sigma(z)=\frac{e^{z}\cdot e^{-z}}{\left(e^{z}+e^{0}\right)\cdot e^{-z}}$",
)
_save_sig2d(fig_s2b, "68_sigmoid_2d_multiply_exp_legend.png")

fig_s2c, ax_s2c = plt.subplots(figsize=SIG2D_FIGSIZE)
_plot_sigmoid_2d_base(ax_s2c)
_legend_sig2d(
    ax_s2c,
    r"$\sigma(z)=\frac{1}{1+e^{-z}}$",
)
_save_sig2d(fig_s2c, "69_sigmoid_2d_standard_legend.png")


### `70_sigmoid_and_mirror.gif`

3D rotation: green **σ(z)** and red **σ(−z)** surfaces.


In [52]:
rotation_azimuths = np.linspace(25, 345, ROTATION_FRAMES)


# Animated 3D rotation around pass/fail sigmoid surfaces (correct ST/EL axes)
frames = []
for az in rotation_azimuths:
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)

    style_sigmoid_axes(ax, az)
    ax.text2D(0.02, 0.95, r"$\sigma(z)=\frac{1}{1+e^{-z}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
    frames.append(fig_to_image(fig))

save_gif(frames, "70_sigmoid_and_mirror.gif", duration=ROTATION_MS)


### `71_sigmoid_static.png`

Single-frame snapshot with formula overlay.


In [53]:
# Static 3D snapshot
fig = plt.figure(figsize=EXPORT_FIGSIZE)
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
style_sigmoid_axes(ax, 58)
ax.text2D(0.02, 0.95, r"$\sigma(z)=\frac{1}{1+e^{-z}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
save_fig(fig, "71_sigmoid_static.png")


### `72_sigmoid_pass_green.gif`

Flat plane → **σ(z)** pass surface, then orbit.


In [59]:
# 1) 61 — flat z=0 plane + class points, bend up to σ(z), then one full 360° orbit (half-degree steps)
frames = []
_az0 = float(extra_angles[0])
Z_flat = np.zeros_like(ST, dtype=float)
zeros_pts = np.zeros_like(all_diff, dtype=float)
for _ in range(SIG3D_FLAT_HOLD):
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Z_flat, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, zeros_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, _az0, diag_prob_scale=0.0)
    frames.append(sig3d_frame_png(fig))
for i in range(SIG3D_MORPH_FRAMES):
    u = i / (SIG3D_MORPH_FRAMES - 1) if SIG3D_MORPH_FRAMES > 1 else 1.0
    su = _sig2d_smoothstep(u)
    Zs = su * P_PASS
    zpt = su * _z_pass_pts
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Zs, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, zpt, rotate_icons_180=True)
    style_sigmoid_axes(ax, _az0, diag_prob_scale=su)
    frames.append(sig3d_frame_png(fig))
for az in extra_angles:
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_PASS, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, _z_pass_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, az)
    frames.append(sig3d_frame_png(fig))
save_gif(frames, "72_sigmoid_pass_green.gif", duration=ROTATION_MS)


### `73_sigmoid_fail_red.gif`

Morph pass surface to fail **σ(−z)**, then orbit.


In [60]:
# 2) 62 — hold pass surface, morph σ(z)→σ(−z) green→red, then one full 360° orbit
_pass_rgb = np.array(to_rgb(SIG_PASS_COLOR))
_fail_rgb = np.array(to_rgb(SIG_FAIL_COLOR))
_SIG61_HOLD_LAST = 14
_SIG61_MORPH = 84

frames = []
for _ in range(_SIG61_HOLD_LAST):
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_PASS, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, _z_pass_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, _last_az_pass)
    frames.append(sig3d_frame_png(fig))
for i in range(_SIG61_MORPH):
    u = i / (_SIG61_MORPH - 1) if _SIG61_MORPH > 1 else 1.0
    su = _sig2d_smoothstep(u)
    Z_surf = (1.0 - su) * P_PASS + su * P_FAIL
    c_surf = tuple((1.0 - su) * _pass_rgb + su * _fail_rgb)
    z_pts = (1.0 - su) * _z_pass_pts + su * _z_fail_pts
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Z_surf, alpha=0.26, color=c_surf, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, z_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, _last_az_pass)
    frames.append(sig3d_frame_png(fig))
for az in extra_angles:
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.26, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
    scatter_whole_data_by_class(ax, _z_fail_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, az)
    frames.append(sig3d_frame_png(fig))
save_gif(frames, "73_sigmoid_fail_red.gif", duration=ROTATION_MS)


### `74_sigmoid_pass_colormap.gif`

Colormap surface morph from flat, then orbit.


In [61]:
# 3) 63 — flat z=0 colormap surface, bend to σ(ST−EL), then one full 360° orbit
frames = []
_az63 = float(extra_angles[0])
for _ in range(SIG3D_FLAT_HOLD):
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Z_flat, alpha=0.32, cmap=CMAP, vmin=0, vmax=1, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, zeros_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, _az63, diag_prob_scale=0.0)
    frames.append(sig3d_frame_png(fig))
for i in range(SIG3D_MORPH_FRAMES):
    u = i / (SIG3D_MORPH_FRAMES - 1) if SIG3D_MORPH_FRAMES > 1 else 1.0
    su = _sig2d_smoothstep(u)
    Zc = su * P_PASS
    zpt = su * _z_pass_pts
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Zc, alpha=0.32, cmap=CMAP, vmin=0, vmax=1, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, zpt, rotate_icons_180=True)
    style_sigmoid_axes(ax, _az63, diag_prob_scale=su)
    frames.append(sig3d_frame_png(fig))
for az in extra_angles:
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        ST,
        EL,
        P_PASS,
        alpha=0.32,
        cmap=CMAP,
        vmin=0,
        vmax=1,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    scatter_whole_data_by_class(ax, _z_pass_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, az)
    frames.append(sig3d_frame_png(fig))
save_gif(frames, "74_sigmoid_pass_colormap.gif", duration=ROTATION_MS)


### `75_sigmoid_colormap_to_topdown.gif`

Tilt to top-down and short azimuth sweep.


In [62]:
# 4) 64 — flat z=0, bend to colormap σ surface; tilt to top-down (half-degree steps);
# fixed left exam-length label near top; strip z ticks/labels once top; then 32° azimuth sweep.
frames = []
_az64 = float(extra_angles[0])
for _ in range(SIG3D_FLAT_HOLD):
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Z_flat, alpha=0.32, cmap=CMAP, vmin=0, vmax=1, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, zeros_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, _az64)
    frames.append(sig3d_frame_png(fig))
for i in range(SIG3D_MORPH_FRAMES):
    u = i / (SIG3D_MORPH_FRAMES - 1) if SIG3D_MORPH_FRAMES > 1 else 1.0
    su = _sig2d_smoothstep(u)
    Zc = su * P_PASS
    zpt = su * _z_pass_pts
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, Zc, alpha=0.32, cmap=CMAP, vmin=0, vmax=1, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, zpt, rotate_icons_180=True)
    _az_morph = float(_az64 + (topdown_azimuth - _az64) * su)
    style_sigmoid_axes(ax, _az_morph)
    frames.append(sig3d_frame_png(fig))

# Phase A: tilt upward (half-degree elevation steps)
for elev in topdown_elevations:
    _near_top = elev >= 87.5
    _exam2d = elev >= 78.0
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        ST,
        EL,
        P_PASS,
        alpha=0.32,
        cmap=CMAP,
        vmin=0,
        vmax=1,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    scatter_whole_data_by_class(ax, _z_pass_pts, rotate_icons_180=True)
    style_sigmoid_axes(
        ax,
        topdown_azimuth,
        elev=float(elev),
        hide_z=_near_top,
        exam_label_2d=_exam2d,
    )
    frames.append(sig3d_frame_png(fig))

# Phase B: at top, rotate counterclockwise 32 degrees (half-degree azimuth steps)
for az in counterclockwise_turn:
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        ST,
        EL,
        P_PASS,
        alpha=0.32,
        cmap=CMAP,
        vmin=0,
        vmax=1,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    scatter_whole_data_by_class(ax, _z_pass_pts, rotate_icons_180=True)
    style_sigmoid_axes(ax, float(az), elev=SIG3D_TOP_ELEV1, hide_z=True, exam_label_2d=True)
    frames.append(sig3d_frame_png(fig))

hold = frames[-1]
for _ in range(TOPDOWN_END_HOLD_FRAMES):
    frames.append(hold.copy())

save_gif(frames, "75_sigmoid_colormap_to_topdown.gif", duration=ROTATION_MS)


### `77_from27_topdown_reveal_lift_3d_reverse64.gif`

**Half-plane reveal** (like **`31`**) on the **amplified** field (**g = 5**) → **2D shrink** (**center fixed**, width eases a bit more than height, **`auto`** aspect) → **3D** top-down (no axis titles) → **−45°** azimuth at top elevation → **tilt** → **360°** spin (makes up **+45°** vs **315°** so final azimuth still **+315°** from pure top) → **de-amplify** toward **~½**, hold, end.


In [63]:
# --- 74 — half-plane → shrink → 3D → −45° az (top) → tilt → spin (360°) → de-amplify → ~½
Z_27 = sigmoid(0.5 * ST - 0.5 * EL)
z_pts_27 = sigmoid(0.5 * all_diff)

def _Z27_gain_surface(g):
    """σ(0.5·g·(ST−EL)); g=1 matches Z_27; g>1 sharpens; g→0 flattens to ~½."""
    return sigmoid(0.5 * float(g) * DIFF)


def _Z27_gain_scatter(g):
    return sigmoid(0.5 * float(g) * all_diff)


# Sharpened field for all of `74` until the post-spin de-amplify (matches former peak gain).
_74_GAIN_HI = 5.0
_74_GAIN_PLAY = float(_74_GAIN_HI)
Z_74 = _Z27_gain_surface(_74_GAIN_PLAY)
z_pts_74 = _Z27_gain_scatter(_74_GAIN_PLAY)
_74_GAIN_LO = 0.06  # de-amplify target: σ(0.5·g·Δ) → ~½ everywhere


# Axis bbox: full 2D (same EXPORT_FIGSIZE / EXPORT_ADJ as other ST–EL colormap plots)
_fig_probe2d, _ax_probe2d = plt.subplots(figsize=EXPORT_FIGSIZE)
_fig_probe2d.subplots_adjust(**EXPORT_ADJ)
_ax_probe2d.contourf(
    ST,
    EL,
    Z_74,
    levels=np.linspace(0.0, 1.0, 45),
    cmap=CMAP,
    vmin=0,
    vmax=1,
    antialiased=True,
    alpha=0.32,
)
add_threshold_line(_ax_probe2d, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
for s, e, lbl in zip(all_study, all_exam, y_real):
    add_outcome_icon(_ax_probe2d, float(s), float(e), passed=bool(lbl), zoom=0.2, alpha=0.95)
_ax_probe2d.set_xlim(*xlim)
_ax_probe2d.set_ylim(*ylim)
_ax_probe2d.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
_ax_probe2d.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
_ax_probe2d.grid(alpha=0.12)
_ax_probe2d.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
add_combined_legend(_ax_probe2d, loc="upper left")
_ax_probe2d.set_aspect("auto")
_fig_probe2d.canvas.draw()
_pos_full_2d = _ax_probe2d.get_position()
plt.close(_fig_probe2d)

_W0_2d = float(_pos_full_2d.width)
_H0_2d = float(_pos_full_2d.height)
_cx_2d = float(_pos_full_2d.x0 + 0.5 * _W0_2d)
_cy_2d = float(_pos_full_2d.y0 + 0.5 * _H0_2d)


# Shrink with x slightly tighter than y (independent end scales), then cross-fade to 3D.
_74_SHRINK_END_SCALE_W = 0.39  # width scale at end (center fixed); lower = narrower
_74_SHRINK_END_SCALE_H = 0.65  # height scale at end


def _shrink_bbox_uniform(u_lin, W0, H0, cx, cy, end_scale_w, end_scale_h):
    """(W0,H0) → (sw·W0, sh·H0) with sw, sh easing from 1 to end_scale_* on one timeline."""
    u_lin = float(np.clip(u_lin, 0.0, 1.0))
    p = _sig2d_smoothstep(_ease_top27(u_lin))
    sw = 1.0 + (float(end_scale_w) - 1.0) * p
    sh = 1.0 + (float(end_scale_h) - 1.0) * p
    w = W0 * sw
    h = H0 * sh
    return mpl.transforms.Bbox.from_bounds(cx - 0.5 * w, cy - 0.5 * h, w, h)


_LIFT2D_SHRINK_FRAMES = 120


def _ease_top27(t):
    t = np.clip(t, 0.0, 1.0)
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _pil_topdown_27style_masked(zm, ax_rect=None):
    fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
    if ax_rect is not None:
        _rw = float(ax_rect.width) / max(_W0_2d, 1e-9)
        _rh = float(ax_rect.height) / max(_H0_2d, 1e-9)
        _rmin = min(_rw, _rh)
        _pad = min(0.16, float(EXPORT_ADJ["left"]) + 0.12 * (1.0 - _rmin))
        _bot = min(0.16, float(EXPORT_ADJ["bottom"]) + 0.10 * (1.0 - _rmin))
        fig.subplots_adjust(
            left=_pad,
            right=float(EXPORT_ADJ["right"]),
            bottom=_bot,
            top=float(EXPORT_ADJ["top"]),
        )
    else:
        fig.subplots_adjust(**EXPORT_ADJ)
    ax.contourf(
        ST,
        EL,
        zm,
        levels=np.linspace(0.0, 1.0, 45),
        cmap=CMAP,
        vmin=0,
        vmax=1,
        antialiased=True,
        alpha=0.32,
    )
    add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
    for s, e, lbl in zip(all_study, all_exam, y_real):
        add_outcome_icon(ax, float(s), float(e), passed=bool(lbl), zoom=0.2, alpha=0.95)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.grid(alpha=0.12)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    add_combined_legend(ax, loc="upper left")
    if ax_rect is not None:
        ax.set_position(ax_rect)
    ax.set_aspect("auto")
    return fig_to_image(fig)


def _strip_74_3d_axis_labels(ax):
    """Remove axis titles (and exam text2D) after `style_sigmoid_axes` — keeps `74` self-contained."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    for t in list(ax.texts):
        try:
            if "Exam length" in t.get_text():
                t.remove()
        except Exception:
            pass


def _frame_3d_z27_surface(
    Z_surf, z_scatter, az, elev, hide_z, exam_label_2d, diag_scale=1.0, hide_axis_labels=True
):
    fig = plt.figure(figsize=SIG3D_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        ST,
        EL,
        Z_surf,
        alpha=0.32,
        cmap=CMAP,
        vmin=0,
        vmax=1,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    scatter_whole_data_by_class(ax, z_scatter, rotate_icons_180=True)
    style_sigmoid_axes(
        ax,
        float(az),
        elev=float(elev),
        hide_z=hide_z,
        exam_label_2d=exam_label_2d,
        diag_prob_scale=diag_scale,
    )
    if hide_axis_labels:
        _strip_74_3d_axis_labels(ax)
    return sig3d_frame_png(fig)


_74_BLEND_FRAMES = 28
_74_DIP_FRAMES = 72
_74_FULL_SPIN_FRAMES = 120
_74_AMP_DOWN_FRAMES = 88
_74_TAIL_HOLD = 10
_74_PRE_TILT_AZ_DEG = -45.0  # azimuth at top elevation before tilt (then orbit makes up +45°)
_74_PRE_TILT_FRAMES = 32
_74_ORBIT_DEG = 360.0  # −45° pre-tilt + 360° spin → same final az as baseline +315° from pure top

_az_top_end = float(counterclockwise_turn[-1])

frames_74 = []

# (1) Half-plane reveal (same cadence as `34_sigmoid_colormap_topdown_reveal.gif`)
_dmax27 = float(np.max(DIFF))
_dmin27 = float(np.min(DIFF))
_eps27 = 0.05 * max(_dmax27, abs(_dmin27))
_NP27, _NF27 = 52, 52
_HOLDPF27, _HOLDA27 = 10, 14

for j in range(_NP27):
    u = _eps27 + (_dmax27 - _eps27) * _ease_top27(j / max(_NP27 - 1, 1))
    zm = np.where((DIFF > 0) & (DIFF <= u), Z_74, np.nan)
    frames_74.append(_pil_topdown_27style_masked(zm))
zm_pass_done = np.where(DIFF > 0, Z_74, np.nan)
for _ in range(_HOLDPF27):
    frames_74.append(_pil_topdown_27style_masked(zm_pass_done))
for j in range(_NF27):
    lo = -_eps27 + (_dmin27 + _eps27) * _ease_top27(j / max(_NF27 - 1, 1))
    zm = np.where(
        DIFF > 0,
        Z_74,
        np.where((DIFF >= lo) & (DIFF < 0), Z_74, np.nan),
    )
    frames_74.append(_pil_topdown_27style_masked(zm))
for _ in range(_HOLDA27):
    frames_74.append(_pil_topdown_27style_masked(Z_74))

# (2) Shrink (x slightly more than y), then 3D top
for j in range(_LIFT2D_SHRINK_FRAMES):
    u_lin = j / max(_LIFT2D_SHRINK_FRAMES - 1, 1)
    rect = _shrink_bbox_uniform(
        u_lin, _W0_2d, _H0_2d, _cx_2d, _cy_2d, _74_SHRINK_END_SCALE_W, _74_SHRINK_END_SCALE_H
    )
    frames_74.append(_pil_topdown_27style_masked(Z_74, ax_rect=rect))

im2d_last = frames_74[-1].copy()
im3d_top = _frame_3d_z27_surface(
    Z_74,
    z_pts_74,
    _az_top_end,
    SIG3D_TOP_ELEV1,
    hide_z=True,
    exam_label_2d=True,
    diag_scale=1.0,
)

# (3) Cross-fade 2D (shrunk) → 3D top-down
_nb = max(_74_BLEND_FRAMES, 2)
for k in range(_nb):
    u = _sig2d_smoothstep(k / float(_nb - 1))
    frames_74.append(Image.blend(im2d_last, im3d_top, u))

# (4) Brief hold on top-down 3D (already at im3d_top; duplicate a few frames for readability)
for _ in range(8):
    frames_74.append(im3d_top.copy())

# (4b) −45° azimuth at top elevation before tilt; orbit adds +45° vs a plain 315° pass (360° total) to match final bearing
_npre = max(_74_PRE_TILT_FRAMES, 2)
for j in range(_npre):
    u = _sig2d_smoothstep(j / float(_npre - 1))
    az = _az_top_end + _74_PRE_TILT_AZ_DEG * u
    frames_74.append(
        _frame_3d_z27_surface(
            Z_74,
            z_pts_74,
            float(az),
            SIG3D_TOP_ELEV1,
            hide_z=True,
            exam_label_2d=True,
            diag_scale=1.0,
        )
    )
_az74_post_top = _az_top_end + _74_PRE_TILT_AZ_DEG

# (5) Smooth tilt: top-down → oblique / side (same thresholds as `72` elevation pass)
_nd = max(_74_DIP_FRAMES, 2)
for j in range(_nd):
    u = _sig2d_smoothstep(j / float(_nd - 1))
    elev = SIG3D_TOP_ELEV1 + (SIG3D_TOP_ELEV0 - SIG3D_TOP_ELEV1) * u
    near_top = elev >= 87.5
    exam2d = elev >= 78.0
    frames_74.append(
        _frame_3d_z27_surface(
            Z_74,
            z_pts_74,
            _az74_post_top,
            float(elev),
            hide_z=near_top,
            exam_label_2d=exam2d,
            diag_scale=1.0,
        )
    )

# (6) One full rotation at side view (elevation fixed at opening angle)
_nspin = max(_74_FULL_SPIN_FRAMES, 1)
for j in range(1, _nspin + 1):
    az = _az74_post_top + _74_ORBIT_DEG * float(j) / float(_nspin)
    frames_74.append(
        _frame_3d_z27_surface(
            Z_74,
            z_pts_74,
            az,
            SIG3D_TOP_ELEV0,
            hide_z=False,
            exam_label_2d=False,
            diag_scale=1.0,
        )
    )

_az_spin_end = _az74_post_top + _74_ORBIT_DEG

# (7) De-amplify after spin: g from G_play down to G_LO so σ → ~½ everywhere

_du_de = max(_74_AMP_DOWN_FRAMES - 1, 1)
for j in range(_74_AMP_DOWN_FRAMES):
    u = _sig2d_smoothstep(j / float(_du_de))
    g = _74_GAIN_PLAY + (_74_GAIN_LO - _74_GAIN_PLAY) * u
    Zg = _Z27_gain_surface(g)
    zg = _Z27_gain_scatter(g)
    frames_74.append(
        _frame_3d_z27_surface(
            Zg,
            zg,
            _az_spin_end,
            SIG3D_TOP_ELEV0,
            hide_z=False,
            exam_label_2d=False,
            diag_scale=1.0,
        )
    )

Z_flat = _Z27_gain_surface(_74_GAIN_LO)
z_flat = _Z27_gain_scatter(_74_GAIN_LO)
for _ in range(_74_TAIL_HOLD):
    frames_74.append(
        _frame_3d_z27_surface(
            Z_flat,
            z_flat,
            _az_spin_end,
            SIG3D_TOP_ELEV0,
            hide_z=False,
            exam_label_2d=False,
            diag_scale=1.0,
        )
    )

save_gif(
    frames_74,
    "77_from27_topdown_reveal_lift_3d_reverse64.gif",
    duration=ROTATION_MS,
)


### `76_sigmoid_colormap_topdown_2d.png`

Static top-down colormap (same field as `74` opening).


In [64]:
# 5) Top-down 2D colormap counterpart (σ(0.5·ST − 0.5·EL); same CMAP / alpha / levels as contour reveal)
fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
Z_sig = sigmoid(0.5 * ST - 0.5 * EL)
ax.contourf(
    ST,
    EL,
    Z_sig,
    levels=np.linspace(0.0, 1.0, 45),
    cmap=CMAP,
    vmin=0,
    vmax=1,
    antialiased=True,
    alpha=0.32,
)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)

for s, e, lbl in zip(all_study, all_exam, y_real):
    add_outcome_icon(ax, s, e, passed=bool(lbl), zoom=0.2, alpha=0.95)

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
ax.grid(alpha=0.12)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "76_sigmoid_colormap_topdown_2d.png")


## Scene 8c — Jar riddle (bacteria doubling)

Matches `script.md`: full on day 1000, then **half** the previous amount each day going backward (`78_jar_riddle_halving.gif`). The **tank** is an **open bar** (left, right, and bottom edges only — **no** top cap). **Water** fills the column from the bottom to the current level. **Days 1000–950** use **four** duplicate frames per day; **days 949–0** use **one** frame per day (**950** days). **No** axis ticks or labels; the legend **Day *n*** matches the modeled day. Legend text uses **2×** `LEGEND_SIZE`.


In [65]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# Bacteria-in-a-jar riddle: full on day 1000; going backward one day at a time halves the volume.
JAR_DAYS = 1000
JAR_MS = 12  # ms per frame
FIG_JAR = EXPORT_FIGSIZE
JAR_Y_MAX = 1000.0
# Open “tank”: half-width of inner water column in data x (centered at 0).
JAR_BAR_HALF_W = 0.28
# Days 1000 … 950 (51 values): a few frames per day. Days 949 … 0 (950 values): one frame per day.
JAR_MULTI_DAY_MIN = 950
JAR_FRAMES_PER_DAY_MULTI = 4
JAR_LEGEND_FONT = LEGEND_SIZE * 2.0

frames_jar = []
x0 = -JAR_BAR_HALF_W
x1 = JAR_BAR_HALF_W
bar_w = x1 - x0
margin = 0.22 * bar_w

for day in range(JAR_DAYS, -1, -1):
    n_half = JAR_DAYS - day
    h = float(JAR_Y_MAX * (0.5**n_half))
    n_rep = JAR_FRAMES_PER_DAY_MULTI if day >= JAR_MULTI_DAY_MIN else 1
    for _ in range(n_rep):
        fig, ax = plt.subplots(figsize=FIG_JAR)
        ax.set_xlim(x0 - margin, x1 + margin)
        ax.set_ylim(0.0, JAR_Y_MAX)

        # Water fills the bar from the bottom up to height h (everything inside the column is water).
        ax.add_patch(
            Rectangle(
                (x0, 0.0),
                bar_w,
                h,
                facecolor="#4f8fc9",
                edgecolor="none",
                zorder=1,
            )
        )

        # Open-top tank: left, right, and bottom edges only (no line across the top).
        _lw = 2.85
        ax.plot([x0, x0], [0.0, JAR_Y_MAX], color="#141414", linewidth=_lw, solid_capstyle="projecting", zorder=3)
        ax.plot([x1, x1], [0.0, JAR_Y_MAX], color="#141414", linewidth=_lw, solid_capstyle="projecting", zorder=3)
        ax.plot([x0, x1], [0.0, 0.0], color="#141414", linewidth=_lw, solid_capstyle="projecting", zorder=3)

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.tick_params(axis="both", which="both", length=0, width=0, labelleft=False, labelbottom=False)
        ax.set_xlabel("")
        ax.set_ylabel("")
        for _sp in ax.spines.values():
            _sp.set_visible(False)
        ax.grid(False)

        leg_h = [
            Line2D(
                [],
                [],
                linestyle="none",
                marker="",
                markersize=0,
                label=f"Day {day}",
            ),
        ]
        ax.legend(
            handles=leg_h,
            loc="upper right",
            frameon=True,
            prop={"size": JAR_LEGEND_FONT},
        )
        fig.subplots_adjust(left=0.12, right=0.96, top=0.96, bottom=0.06)
        frames_jar.append(fig_to_image(fig, tight_layout=False))
        plt.close(fig)

save_gif(frames_jar, "78_jar_riddle_halving.gif", duration=JAR_MS)


## Scene 8d — Logistic gradient descent + three-class softmax

**Prerequisites:** Cell 1, Cells 2–3 (datasets, `xlim` / `ylim`). **Export 79** — run the first code cell only. **Export 80** — run **Scene 8 — Setup (shared)** first, then the second code cell (`ST`, `EL`, `style_sigmoid_axes`, `sig3d_frame_png`, `ROTATION_FRAMES`, `ROTATION_MS`).

**Exports**
- **`79_logistic_gradient_descent_plane.gif`** — **`tensorflow.keras`** **`Sequential`**: **`Dense(1, input_dim=2, activation="sigmoid")`** (weights **2×1** + bias), **`binary_crossentropy`**, **Adam**; same **snap → `fig_to_image` → `save_gif`** loop as your pasted **`LR`** class (fit chunks, then predict on a fine mesh). Training points are the **full realistic** roster (**`study_real`**, **`exam_real`**, **`y_real`** from Cells 2–3: separable set plus noisy symmetric adds). **`EXPORT_FIGSIZE`** / **`EXPORT_ADJ`**, **`CMAP_GD`**, **`draw_dataset`** for check/cross icons; output **`renders/79_...gif`** with duplicate frames + tail hold.
- **`80_softmax_three_class_rotate.gif`** — Synthetic **three-class** points (**blue**, **yellow**, **purple**) with matching **softmax** probability **surfaces**; scatter at each point’s **true-class** probability; **360°** orbit (same cadence as **`70_sigmoid_and_mirror.gif`**). No equation or legend; **ST**/**EL** limits are **data min/max ± 0.5**.


In [7]:
# Scene 8d (1/2) — Keras single-layer logistic (2 → 1, sigmoid) + binary CE → export 79
# Full **realistic** roster from Cells 2–3 (`study_real`, `exam_real`, `y_real`). Same LR snap / animate as your paste.
# Export canvas: EXPORT_FIGSIZE / EXPORT_ADJ + fig_to_image (no bbox tight). CMAP_GD + draw_dataset.

import matplotlib as mpl
import subprocess
import sys

# TensorFlow must live in the same environment as the Jupyter kernel. If `pip install`
# used another Python, imports fail — install into *this* interpreter via sys.executable.
try:
    import tensorflow as tf  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "tensorflow>=2.15"],
        stdout=sys.stdout,
        stderr=sys.stderr,
    )
    import tensorflow as tf  # noqa: F401

from tensorflow import keras

GD_MS = 40
GD_HOLD_LAST = 18

cvals = [0, 0.5, 1]
colors = ["red", "white", "green"]
_gd_norm = plt.Normalize(min(cvals), max(cvals))
_gd_tuples = list(zip(map(_gd_norm, cvals), colors))
CMAP_GD = mpl.colors.LinearSegmentedColormap.from_list("gd_sigmoid_plane", _gd_tuples, 100)

_xa, _xb = float(xlim[0]), float(xlim[1])
_ya, _yb = float(ylim[0]), float(ylim[1])
_h = 0.02
_gd_st = np.arange(_xa, _xb + 1e-9, _h)
_gd_el = np.arange(_ya, _yb + 1e-9, _h)
ST_gd, EL_gd = np.meshgrid(_gd_st, _gd_el)
meshData = np.c_[ST_gd.ravel(), EL_gd.ravel()].astype(np.float32)

# Cells 2–3: `realistic_points` = separable + noisy; unpacked as study_real, exam_real, y_real.
X = np.column_stack([study_real.astype(np.float32), exam_real.astype(np.float32)])
Y = y_real.astype(int)
Y_0_1 = Y.reshape(-1, 1).astype(np.float32)


class LR:
    def __init__(self, model):
        self.model = model

    def _snap_learning(self, X, Y, filename=None):
        fig, ax = plt.subplots(figsize=EXPORT_FIGSIZE)
        fig.subplots_adjust(**EXPORT_ADJ)
        Z = self.model.predict(meshData, verbose=0).reshape(ST_gd.shape)
        ax.contourf(ST_gd, EL_gd, Z, alpha=0.4, cmap=CMAP_GD, vmin=0, vmax=1, zorder=1)
        draw_dataset(ax, study_real, exam_real, y_real)
        ax.set_xlim(_xa, _xb)
        ax.set_ylim(_ya, _yb)
        ax.grid(alpha=0.2)
        return fig_to_image(fig, tight_layout=False)

    def animate_learning(self, X, Y, Y_0_1, snap_freq=10, filename="learn", duration=1000, **kwargs):
        """Fit in chunks of `snap_freq` epochs, snap after each chunk. `Y` = integer labels for icons."""
        images = []
        if "epochs" in kwargs:
            epochs = kwargs["epochs"]
            kwargs.pop("epochs", None)
        else:
            epochs = snap_freq
        for _ in range(int(epochs / snap_freq)):
            self.model.fit(X, Y_0_1, epochs=snap_freq, verbose=0, **kwargs)
            images.append(self._snap_learning(X, Y, filename))
        return images


model = keras.models.Sequential()
model.add(keras.layers.Dense(1, input_dim=2, activation="sigmoid"))
model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=1e-1))

obj = LR(model)
frames_gd = obj.animate_learning(X, Y, Y_0_1, snap_freq=1, filename="049", duration=GD_MS, epochs=40, batch_size=100)

out = []
for im in frames_gd:
    out.append(im.copy())
    out.append(im.copy())
last = frames_gd[-1].copy()
for _ in range(GD_HOLD_LAST):
    out.append(last.copy())

save_gif(out, "79_logistic_gradient_descent_plane.gif", duration=GD_MS)


In [79]:
# Scene 8d (2/2) — three-class softmax 3D orbit → export 80 (needs Scene 8 Setup)

# --- 80: three-class softmax surfaces + scatter; full 360° orbit (needs Scene 8 setup) ---


def _softmax_rows(logits_mat):
    logits_mat = np.asarray(logits_mat, dtype=float)
    logits_mat = logits_mat - np.max(logits_mat, axis=1, keepdims=True)
    ex = np.exp(logits_mat)
    return ex / np.sum(ex, axis=1, keepdims=True)


rng_sm = np.random.default_rng(202)
CLASS_COLORS = ("#1f55af", "#f4d03f", "#7b2cbf")
centers_sm = [(3.2, 4.1, 0), (6.0, 3.6, 1), (5.0, 6.4, 2)]
n_per_c = 6
xs_sm, ys_sm, yc_sm = [], [], []
for cx, cy, ck in centers_sm:
    for _ in range(n_per_c):
        xs_sm.append(np.clip(cx + 0.42 * rng_sm.standard_normal(), xlim[0] + 0.08, xlim[1] - 0.08))
        ys_sm.append(np.clip(cy + 0.42 * rng_sm.standard_normal(), ylim[0] + 0.08, ylim[1] - 0.08))
        yc_sm.append(ck)
xs_sm = np.asarray(xs_sm, dtype=float)
ys_sm = np.asarray(ys_sm, dtype=float)
yc_sm = np.asarray(yc_sm, dtype=int)
n_sm = len(xs_sm)
X_sm = np.column_stack([np.ones(n_sm), xs_sm, ys_sm])
Y_oh = np.zeros((n_sm, 3), dtype=float)
Y_oh[np.arange(n_sm), yc_sm] = 1.0

W_sm = np.random.randn(3, 3) * 0.06
lr_sm = 0.45
for _ in range(500):
    logits = X_sm @ W_sm.T
    probs = _softmax_rows(logits)
    grad_W = (probs - Y_oh).T @ X_sm / float(n_sm)
    W_sm = W_sm - lr_sm * grad_W

_sm_xlim = (float(np.min(xs_sm)) - 0.5, float(np.max(xs_sm)) + 0.5)
_sm_ylim = (float(np.min(ys_sm)) - 0.5, float(np.max(ys_sm)) + 0.5)
# Same pattern as 002-logistic-regression-part-2 softmax GIF: mesh and limits use final_xlim / final_ylim only
# (avoid style_sigmoid_axes, which pins axes to global Scene-8 xlim/ylim and draws a long diagonal).
final_xlim = [float(_sm_xlim[0]), float(_sm_xlim[1])]
final_ylim = [float(_sm_ylim[0]), float(_sm_ylim[1])]
_sm_plot_n = 240
_st_sm = np.linspace(final_xlim[0], final_xlim[1], _sm_plot_n)
_el_sm = np.linspace(final_ylim[0], final_ylim[1], _sm_plot_n)
ST_sm, EL_sm = np.meshgrid(_st_sm, _el_sm)
logits_grid = np.stack(
    [W_sm[k, 0] + W_sm[k, 1] * ST_sm + W_sm[k, 2] * EL_sm for k in range(3)],
    axis=0,
)
lg = logits_grid.reshape(3, -1).T
P_grid = _softmax_rows(lg).T.reshape(3, ST_sm.shape[0], ST_sm.shape[1])

orbit_az = np.linspace(25.0, 25.0 + 360.0, ROTATION_FRAMES, endpoint=False)
frames_sm = []
for az in orbit_az:
    fig = plt.figure(figsize=EXPORT_FIGSIZE)
    ax = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=26, azim=float(az))
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=12.5)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=12.5)
    ax.set_zlabel("Probability", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.tick_params(axis="x", labelsize=FONT_SIZE)
    ax.tick_params(axis="y", labelsize=FONT_SIZE)
    ax.tick_params(axis="z", labelsize=FONT_SIZE)
    ax.set_xlim([final_xlim[0], final_xlim[1]])
    ax.set_ylim([final_ylim[0], final_ylim[1]])
    ax.set_zlim(0, 1)
    for k in range(3):
        ax.plot_surface(
            ST_sm,
            EL_sm,
            P_grid[k],
            rstride=2,
            cstride=2,
            color=CLASS_COLORS[k],
            linewidth=0,
            antialiased=True,
            alpha=0.38,
            shade=False,
        )
    for i in range(n_sm):
        kk = int(yc_sm[i])
        log_i = W_sm @ np.array([1.0, xs_sm[i], ys_sm[i]])
        pr_i = _softmax_rows(log_i.reshape(1, -1))[0, kk]
        ax.scatter(
            [xs_sm[i]],
            [ys_sm[i]],
            [float(pr_i)],
            color=CLASS_COLORS[kk],
            s=110,
            depthshade=True,
            edgecolors="#1a1a1a",
            linewidths=0.45,
            zorder=6,
        )
    frames_sm.append(sig3d_frame_png(fig))

save_gif(frames_sm, "80_softmax_three_class_rotate.gif", duration=ROTATION_MS)


> **Scene 8:** sigmoid exports are split across **Setup** + one cell per file (59–74, 73). Run **Cell 1** and Scene 8 Setup, then individual cells. **Scene 8d:** first code cell (after datasets) exports **`79_logistic_gradient_descent_plane.gif`**; run **Scene 8 Setup** then the second code cell for **`80_softmax_three_class_rotate.gif`**.

## Scene 9 - Run all exports in order

Execute all cells top-to-bottom once. The `renders/` folder will contain:

Survey table **`00_student_survey_table.gif`**: header row **Student / Exam length / Study time / Pass / fail**, then each **realistic** student's values and ✓/✗ filled **one cell at a time**.

Intro (survey **`00_student_survey_table.gif`**, then **`01`–`17`**): **`01`/`02`** hold empty axes while emphasizing **Study time** then **Exam length** on the axis labels; `03_axes_only.png` is both labels regular weight; `04_dataset_build.gif` holds no points briefly, then adds one point at a time in order `(3,2,1)` then `(2,3,0)` then shuffled rest, with many more frames on the first two points; fail point crosses unseen boundary; threshold shading/labels progression. Then **`18_dataset_full_realistic_st_el_0.png`** / **`19_dataset_full_realistic_st_el_0_marker_33.png`** show the **full** dataset (all examples) with **ST - EL = 0** in the legend; the second adds a **black** marker on the boundary at **(3, 3)**.

After **`31_parallel_lines_additive.gif`**, **`32_fail_point_crosses_threshold_st_el_0.gif`** repeats the **`04`** motion (fail example slides right at fixed exam length) with **ST - EL = 0** drawn.

Scene 6b (**`38_dataset_plane_into_norm_left_slot.gif`** through **`44_norm_exp_bar_hint.gif`** / **`44b_norm_all_st_el_then_exp_emphasis.gif`**): the bridge GIF shrinks the study–exam plane (**ST − EL = 0** threshold only) into the duo left slot; plane + vertical **ST − EL** bars (greens / red); **`40_norm_positive_stack.gif`** positive stack with **$1/(1{+}4)\!\to\!1/5\!\to\!20\%$**; **`41_norm_negative_invalid.gif`**, **`41b_dataset_plane_2h_difference_segment.gif`**, **`41c_dataset_plane_zero_study_exam_gap_contrast.gif`**, **`42_norm_abs_pitfall.gif`** GIFs step through **$(-1,4)$** and **$|z|$** proportion fallacies; **`43_norm_abs_bar_hint.gif`/`44_norm_exp_bar_hint.gif`** bar walkthroughs on **$z\in[-3,3]$** (**$|z|$** vs **$2^{z}$**); same **red–white–green** colormap as sigmoid **3D**.

Scene 5 (`33_not_realistic_transition.gif`–`36_proportions_60_80_100.gif`): `33_not_realistic_transition.gif` holds the **separable** students plus threshold, then adds only the **noisy** students one at a time (**(4,3)** first, then **(3,4)**, then the rest), with axis ticks/labels and legend like the other dataset views. **`34_sigmoid_colormap_topdown_reveal.gif`** animates the top-down **σ(0.5·ST − 0.5·EL)** field (half-plane reveal). **`35_parallel_lines_additive_realistic.gif`** replays the **`31_parallel_lines_additive.gif`**-style additive parallel lines on the **realistic** plane (**`ST - EL = 0`** labeled only; guides **`1, -1, 2, 3`**). `36_proportions_60_80_100.gif` layers the same lines in that order with pass-rate text (**+1** still uses the same counting cadence for **60%**).

1. `00_student_survey_table.gif`
2. `01_axes_study_time_bold.gif`
3. `02_axes_exam_length_bold.gif`
4. `03_axes_only.png`
5. `04_dataset_build.gif`
6. `05_fail_point_crosses_threshold.gif`
7. `06_threshold_unlabeled.png`
8. `07_threshold_green_right.png`
9. `08_threshold_red_left.png`
10. `09_threshold_micro_drift.gif`
11. `10_dataset_extra_pass_shifted_threshold.png`
12. `11_dataset_two_extra_no_threshold.png`
13. `12_threshold_labeled_st_eq_el.png`
14. `13_threshold_st_eq_el_dot_33.png`
15. `14_threshold_green_labeled_st_gt_el.png`
16. `15_threshold_red_labeled_st_lt_el.png`
17. `16_threshold_label_st_minus_el_eq_el_minus_el.png`
18. `17_threshold_label_st_minus_el_eq_0.png`
19. `18_dataset_full_realistic_st_el_0.png`
20. `19_dataset_full_realistic_st_el_0_marker_33.png`
21. `20_dataset_overview.png`
22. `21_dataset_overview_labeled.png`
23. `22_dataset_points_on_threshold.gif`
24. `23_threshold_green_st_minus_el_gt_0.png`
25. `24_threshold_red_st_minus_el_lt_0.png`
26. `25_dataset_st2_el1_marker_ticks.gif`
27. `26_parallel_lines_step_01.png`
28. `27_parallel_lines_step_02.png`
29. `28_parallel_lines_step_03.png`
30. `29_parallel_lines_step_04.png`
31. `30_parallel_lines_step_05.png`
32. `31_parallel_lines_additive.gif`
33. `32_fail_point_crosses_threshold_st_el_0.gif`
34. `33_not_realistic_transition.gif`
35. `34_sigmoid_colormap_topdown_reveal.gif`
36. `35_parallel_lines_additive_realistic.gif`
37. `36_proportions_60_80_100.gif`
38. `37_between_60_and_80.png`
39. `38_dataset_plane_into_norm_left_slot.gif`
40. `39_norm_positive_two_bars.png`
41. `40_norm_positive_stack.gif`
42. `41_norm_negative_invalid.gif`
43. `41b_dataset_plane_2h_difference_segment.gif`
44. `41c_dataset_plane_zero_study_exam_gap_contrast.gif`
45. `42_norm_abs_pitfall.gif`
46. `43_norm_abs_bar_hint.gif`
47. `44_norm_exp_bar_hint.gif`
48. `44b_norm_all_st_el_then_exp_emphasis.gif`
49. `45_exponential_point_guides.gif`
50. `46_exponential_mapping.gif`
51. `47_step_left_drop_one_to_zero.png`
52. `48_step_two_drop_half_to_zero.png`
53. `49_exponential_transition.gif`
54. `50_exponential_2x_legend.png`
55. `51_exponential_ex_legend.png`
56. `52_two_pow_axis_ln2_to_exp.gif`
57. `53_norm_exp_positive_softmax.gif`
58. `54_norm_exp_negative_softmax.gif`
59. `55_far_study_hours_zoom.gif`
60. `56_softmax_threshold_and_z1.gif`
61. `57_softmax_pf_stack_normalized_frac.png`
62. `58_softmax_pf_stack_fail_sum_ratio.png`
63. `59_softmax_pf_stack_fail_one_minus.png`
64. `60_softmax_pf_stack_fail_eneg1.png`
65. `61_softmax_pf_stack_fail_sum_ratio_pass_B.png`
66. `62_sigmoid_2d_normalized_exp_reveal.gif`
67. `66_sigmoid_sigma_negz_reveal.gif`
68. `67_sigmoid_2d_normalized_exp_legend.png`
69. `68_sigmoid_2d_multiply_exp_legend.png`
70. `69_sigmoid_2d_standard_legend.png`
71. `70_sigmoid_and_mirror.gif`
72. `71_sigmoid_static.png`
73. `72_sigmoid_pass_green.gif`
74. `73_sigmoid_fail_red.gif`
75. `74_sigmoid_pass_colormap.gif`
76. `75_sigmoid_colormap_to_topdown.gif`
77. `76_sigmoid_colormap_topdown_2d.png`
78. `77_from27_topdown_reveal_lift_3d_reverse64.gif`
79. `78_jar_riddle_halving.gif`
80. `79_logistic_gradient_descent_plane.gif`
81. `80_softmax_three_class_rotate.gif`
